In [ ]:
# Cell 1: Environment setup
# Installs/checks packages needed for ingestion. No `kaggle` package needed
# since both datasets are already present in Drive.

import subprocess
import sys


def pip_install(spec: str) -> None:
    """Install a package quietly via pip, raising on failure."""
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", spec],
        check=True,
    )


# Core scientific / ML stack (torch, pandas, sklearn usually preinstalled
# in Colab, but we pin/verify versions here rather than assume).
for pkg in [
    "pandas>=2.0.0",
    "scikit-learn>=1.3.0",
    "sentence-transformers>=2.2.2",
    "transformers>=4.35.0",
    "torch>=2.0.0",
    "pypdf>=3.17.0",  # only for the optional PDF spot-check
]:
    pip_install(pkg)

import pandas as pd
import sklearn
import torch
import transformers
import sentence_transformers
import pypdf

print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("pypdf:", pypdf.__version__)

pandas: 2.2.3
scikit-learn: 1.6.1
torch: 2.11.0+cu128
transformers: 5.16.1
sentence-transformers: 5.7.0
pypdf: 6.19.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Cell 2: Project paths
# Assumes Drive is already mounted at /content/drive.

import os

PROJECT_ROOT = "/content/drive/MyDrive/bias_detection_project"

RAW_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
REPORTS_DIR = os.path.join(PROJECT_ROOT, "reports")
CODE_DIR = os.path.join(PROJECT_ROOT, "code")

KAGGLE_DIR = os.path.join(RAW_DIR, "kaggle")
KAGGLE_CSV = os.path.join(KAGGLE_DIR, "Resume", "Resume.csv")
KAGGLE_PDF_DIR = os.path.join(KAGGLE_DIR, "data")

FAIRCV_DIR = os.path.join(RAW_DIR, "faircvdb")
FAIRCV_NPY = os.path.join(FAIRCV_DIR, "FairCVdb.npy")
FAIRCV_PY = os.path.join(FAIRCV_DIR, "FairCV.py")

# Only create processed/reports/code dirs — raw/ is left untouched since
# it already holds the real data.
for d in (PROCESSED_DIR, REPORTS_DIR, CODE_DIR):
    os.makedirs(d, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Processed dir ready:", PROCESSED_DIR)
print("Reports dir ready:", REPORTS_DIR)
print("Code dir ready:", CODE_DIR)

PROJECT_ROOT: /content/drive/MyDrive/bias_detection_project
Processed dir ready: /content/drive/MyDrive/bias_detection_project/data/processed
Reports dir ready: /content/drive/MyDrive/bias_detection_project/reports
Code dir ready: /content/drive/MyDrive/bias_detection_project/code


In [ ]:
# Cell 3: Verify Kaggle resume dataset is present and spot-check PDFs

import random
import pandas as pd

if not os.path.exists(KAGGLE_CSV):
    print(
        f"[MISSING] Resume.csv not found at expected path:\n  {KAGGLE_CSV}\n"
        "Place the Kaggle 24-class Resume dataset CSV there before continuing."
    )
else:
    kaggle_df = pd.read_csv(KAGGLE_CSV)
    print(f"Loaded Resume.csv: {len(kaggle_df)} rows")
    print("Unique Category values:")
    print(sorted(kaggle_df["Category"].unique().tolist()))

    # Spot-check ~20 random IDs against the PDF folder structure.
    if not os.path.isdir(KAGGLE_PDF_DIR):
        print(f"[WARNING] PDF data folder not found at: {KAGGLE_PDF_DIR}")
    else:
        sample_n = min(20, len(kaggle_df))
        sample_rows = kaggle_df.sample(n=sample_n, random_state=42)

        mismatches = []
        for _, row in sample_rows.iterrows():
            resume_id = row["ID"]
            category = row["Category"]
            expected_pdf = os.path.join(
                KAGGLE_PDF_DIR, str(category), f"{resume_id}.pdf"
            )
            if not os.path.isfile(expected_pdf):
                mismatches.append((resume_id, category, expected_pdf))

        if mismatches:
            print(f"[WARNING] {len(mismatches)}/{sample_n} spot-checked IDs "
                  "have no matching PDF:")
            for resume_id, category, path in mismatches:
                print(f"  ID={resume_id} Category={category} expected={path}")
        else:
            print(f"Spot-check passed: {sample_n}/{sample_n} sampled IDs "
                  "have matching PDFs.")

Loaded Resume.csv: 2484 rows
Unique Category values:
['ACCOUNTANT', 'ADVOCATE', 'AGRICULTURE', 'APPAREL', 'ARTS', 'AUTOMOBILE', 'AVIATION', 'BANKING', 'BPO', 'BUSINESS-DEVELOPMENT', 'CHEF', 'CONSTRUCTION', 'CONSULTANT', 'DESIGNER', 'DIGITAL-MEDIA', 'ENGINEERING', 'FINANCE', 'FITNESS', 'HEALTHCARE', 'HR', 'INFORMATION-TECHNOLOGY', 'PUBLIC-RELATIONS', 'SALES', 'TEACHER']
[WARNING] 8/20 spot-checked IDs have no matching PDF:
  ID=17562754 Category=DIGITAL-MEDIA expected=/content/drive/MyDrive/bias_detection_project/data/raw/kaggle/data/DIGITAL-MEDIA/17562754.pdf
  ID=30311725 Category=CONSTRUCTION expected=/content/drive/MyDrive/bias_detection_project/data/raw/kaggle/data/CONSTRUCTION/30311725.pdf
  ID=19007667 Category=CHEF expected=/content/drive/MyDrive/bias_detection_project/data/raw/kaggle/data/CHEF/19007667.pdf
  ID=17199951 Category=DESIGNER expected=/content/drive/MyDrive/bias_detection_project/data/raw/kaggle/data/DESIGNER/17199951.pdf
  ID=62312955 Category=DESIGNER expected=/co

In [ ]:
# Cell 4: Verify FairCVdb is present and sanity-check its structure

import numpy as np

if not os.path.isfile(FAIRCV_NPY):
    print(
        f"[MISSING] FairCVdb.npy not found at expected path:\n  {FAIRCV_NPY}\n"
        "Place the FairCVdb .npy file (from BiDAlab/FairCVtest) there."
    )
elif not os.path.isfile(FAIRCV_PY):
    print(
        f"[MISSING] FairCV.py reference script not found at:\n  {FAIRCV_PY}\n"
        "Place BiDAlab's FairCV.py loading script there before continuing."
    )
else:
    faircv_dict = np.load(FAIRCV_NPY, allow_pickle=True).item()
    print("Top-level keys in FairCVdb.npy:")
    for k in faircv_dict.keys():
        print(" -", k)

    n_train = len(faircv_dict["Profiles Train"])
    n_test = len(faircv_dict["Profiles Test"])
    print(f"\n'Profiles Train' rows: {n_train} (expected 19200)")
    print(f"'Profiles Test' rows:  {n_test} (expected 4800)")

    if n_train != 19200:
        print("[WARNING] Train row count does not match expected 19,200.")
    if n_test != 4800:
        print("[WARNING] Test row count does not match expected 4,800.")

Top-level keys in FairCVdb.npy:
 - Profiles Train
 - Profiles Test
 - Bios Train
 - Bios Test
 - Names Train
 - Names Test
 - Blind Labels Train
 - Blind Labels Test
 - Biased Labels Train (Gender)
 - Biased Labels Test (Gender)
 - Biased Labels Train (Ethnicity)
 - Biased Labels Test (Ethnicity)
 - Image List Train
 - Image List Test

'Profiles Train' rows: 19200 (expected 19200)
'Profiles Test' rows:  4800 (expected 4800)


In [ ]:
%%writefile /content/drive/MyDrive/bias_detection_project/code/resume_ingestor.py
"""
resume_ingestor.py

Ingests two heterogeneous HR datasets (Kaggle 24-class resume dataset and
FairCVdb) into a single unified DataFrame schema for downstream bias
detection processing.
"""

from __future__ import annotations

import os
from typing import Dict, List, Optional

import numpy as np
import pandas as pd


# Positional decode maps for FairCVdb's 'Profiles' arrays.
ETHNICITY_MAP: Dict[int, str] = {0: "East Asian", 1: "Sub-Saharan African and South Asian", 2: "Caucasian"}
GENDER_MAP: Dict[int, str] = {0: "Male", 1: "Female"}
OCCUPATION_MAP: Dict[int, str] = {
    0: "nurse",
    1: "surgeon",
    2: "physician",
    3: "journalist",
    4: "photographer",
    5: "filmmaker",
    6: "teacher",
    7: "professor",
    8: "attorney",
    9: "accountant",
}

# Expected dataset sizes, used only for a soft validation warning.
EXPECTED_KAGGLE_ROWS = 2_484
EXPECTED_FAIRCV_TRAIN_ROWS = 19_200
EXPECTED_FAIRCV_TEST_ROWS = 4_800

UNIFIED_COLUMNS: List[str] = [
    "resume_id",
    "raw_text",
    "category_label",
    "bio_text",
    "name",
    "gender_label",
    "ethnicity_label",
    "occupation_label",
    "suitability_score",
    "blind_score",
    "biased_score_gender",
    "biased_score_ethnicity",
    "source_dataset",
]


class ResumeIngestor:
    """Ingests Kaggle resume data and FairCVdb into one common schema."""

    def ingest_kaggle(self, csv_path: str) -> pd.DataFrame:
        """Load the Kaggle 24-class Resume dataset CSV.

        Args:
            csv_path: Path to Resume.csv (columns: ID, Resume_str,
                Resume_html, Category).

        Returns:
            DataFrame with columns resume_id, raw_text, category_label,
            source_dataset.

        Raises:
            FileNotFoundError: If csv_path does not exist.
            ValueError: If expected columns are missing.
        """
        if not os.path.isfile(csv_path):
            raise FileNotFoundError(
                f"Kaggle Resume.csv not found at: {csv_path}. "
                "Confirm the dataset is placed under "
                "PROJECT_ROOT/data/raw/kaggle/Resume/Resume.csv."
            )

        df = pd.read_csv(csv_path)

        required_cols = {"ID", "Resume_str", "Category"}
        missing_cols = required_cols - set(df.columns)
        if missing_cols:
            raise ValueError(
                f"Resume.csv is missing expected columns: {missing_cols}"
            )

        out = pd.DataFrame(
            {
                "resume_id": df["ID"],
                "raw_text": df["Resume_str"],
                "category_label": df["Category"],
                "source_dataset": "kaggle_resume",
            }
        )

        n_malformed = out["raw_text"].isna().sum()
        if n_malformed:
            print(
                f"[WARNING] {n_malformed} Kaggle rows have missing "
                "Resume_str text."
            )

        return out

    def ingest_faircv(self, npy_path: str) -> pd.DataFrame:
        """Load FairCVdb.npy and unify its Train/Test splits.

        Args:
            npy_path: Path to FairCVdb.npy.

        Returns:
            DataFrame with the FairCVdb-specific unified schema (see
            module docstring / UNIFIED_COLUMNS).

        Raises:
            FileNotFoundError: If npy_path does not exist.
            KeyError: If expected keys are missing from the pickled dict.
        """
        if not os.path.isfile(npy_path):
            raise FileNotFoundError(
                f"FairCVdb.npy not found at: {npy_path}. "
                "Confirm the dataset is placed under "
                "PROJECT_ROOT/data/raw/faircvdb/FairCVdb.npy."
            )

        data = np.load(npy_path, allow_pickle=True).item()

        required_keys = {
            "Profiles Train",
            "Bios Train",
            "Names Train",
            "Blind Labels Train",
            "Biased Labels Train (Gender)",
            "Biased Labels Train (Ethnicity)",
            "Profiles Test",
            "Bios Test",
            "Names Test",
            "Blind Labels Test",
            "Biased Labels Test (Gender)",
            "Biased Labels Test (Ethnicity)",
        }
        missing_keys = required_keys - set(data.keys())
        if missing_keys:
            raise KeyError(
                f"FairCVdb.npy is missing expected keys: {missing_keys}"
            )

        frames = []
        for split in ("Train", "Test"):
            frames.append(self._decode_faircv_split(data, split))

        combined = pd.concat(frames, ignore_index=True)

        n_train = len(data["Profiles Train"])
        n_test = len(data["Profiles Test"])
        total_expected = EXPECTED_FAIRCV_TRAIN_ROWS + EXPECTED_FAIRCV_TEST_ROWS
        if n_train != EXPECTED_FAIRCV_TRAIN_ROWS or n_test != EXPECTED_FAIRCV_TEST_ROWS:
            print(
                f"[WARNING] FairCVdb split sizes (train={n_train}, "
                f"test={n_test}) differ from expected "
                f"(train={EXPECTED_FAIRCV_TRAIN_ROWS}, "
                f"test={EXPECTED_FAIRCV_TEST_ROWS}); total expected "
                f"{total_expected}."
            )

        return combined

    def _decode_faircv_split(
        self, data: Dict[str, np.ndarray], split: str
    ) -> pd.DataFrame:
        """Decode one FairCVdb split ('Train' or 'Test') into the unified schema."""
        profiles = np.asarray(data[f"Profiles {split}"])
        bios = data[f"Bios {split}"]
        names = data[f"Names {split}"]
        blind = data[f"Blind Labels {split}"]
        biased_gender = data[f"Biased Labels {split} (Gender)"]
        biased_ethnicity = data[f"Biased Labels {split} (Ethnicity)"]

        n_rows = profiles.shape[0]
        prefix = "faircv_train" if split == "Train" else "faircv_test"

        records = []
        for i in range(n_rows):
            row = profiles[i]

            ethnicity_code = int(row[0])
            gender_code = int(row[1])
            occupation_code = int(row[2])
            suitability = float(row[3])
            education = row[4]
            experience = row[5]
            recommendation = row[6]
            availability = row[7]
            language_proficiency = row[8:11]

            ethnicity_label = ETHNICITY_MAP.get(ethnicity_code, "Unknown")
            gender_label = GENDER_MAP.get(gender_code, "Unknown")
            occupation_label = OCCUPATION_MAP.get(occupation_code, "Unknown")

            raw_text = (
                f"Occupation: {occupation_label}; "
                f"Suitability: {suitability}; "
                f"Education: {education}; "
                f"Experience: {experience}; "
                f"Recommendation: {recommendation}; "
                f"Availability: {availability}; "
                f"Language proficiency: {list(language_proficiency)}"
            )

            records.append(
                {
                    "resume_id": f"{prefix}_{i}",
                    "raw_text": raw_text,
                    "category_label": np.nan,
                    "bio_text": bios[i],
                    "name": names[i],
                    "gender_label": gender_label,
                    "ethnicity_label": ethnicity_label,
                    "occupation_label": occupation_label,
                    "suitability_score": suitability,
                    "blind_score": blind[i],
                    "biased_score_gender": biased_gender[i],
                    "biased_score_ethnicity": biased_ethnicity[i],
                    "source_dataset": "faircv",
                }
            )

        return pd.DataFrame(records)

    def ingest_all(
        self, kaggle_csv_path: str, faircv_npy_path: str
    ) -> pd.DataFrame:
        """Ingest both datasets and concatenate into one unified schema.

        Args:
            kaggle_csv_path: Path to Resume.csv.
            faircv_npy_path: Path to FairCVdb.npy.

        Returns:
            A single DataFrame with UNIFIED_COLUMNS, NaN-filled where a
            field doesn't apply to a given source.
        """
        kaggle_df = self.ingest_kaggle(kaggle_csv_path)
        faircv_df = self.ingest_faircv(faircv_npy_path)

        # Reindex both to the full unified schema so concat aligns cleanly.
        kaggle_full = kaggle_df.reindex(columns=UNIFIED_COLUMNS)
        faircv_full = faircv_df.reindex(columns=UNIFIED_COLUMNS)

        unified = pd.concat([kaggle_full, faircv_full], ignore_index=True)

        n_kaggle = len(kaggle_df)
        n_faircv = len(faircv_df)
        expected_faircv_total = (
            EXPECTED_FAIRCV_TRAIN_ROWS + EXPECTED_FAIRCV_TEST_ROWS
        )

        if n_kaggle != EXPECTED_KAGGLE_ROWS:
            print(
                f"[WARNING] Kaggle row count {n_kaggle} does not match "
                f"expected {EXPECTED_KAGGLE_ROWS}. Dataset version may "
                "have drifted."
            )
        if n_faircv != expected_faircv_total:
            print(
                f"[WARNING] FairCVdb total row count {n_faircv} does not "
                f"match expected {expected_faircv_total}. Dataset version "
                "may have drifted."
            )

        return unified

Writing /content/drive/MyDrive/bias_detection_project/code/resume_ingestor.py


In [ ]:
# Cell 6: Run the ingestor end-to-end and persist the unified dataset

import sys

sys.path.insert(0, CODE_DIR)

from resume_ingestor import ResumeIngestor  # noqa: E402

ingestor = ResumeIngestor()
df = ingestor.ingest_all(
    kaggle_csv_path=KAGGLE_CSV,
    faircv_npy_path=FAIRCV_NPY,
)

output_path = os.path.join(PROCESSED_DIR, "ingested_resumes.csv")
df.to_csv(output_path, index=False)
print(f"Saved unified dataset to: {output_path}\n")

print("df.shape:", df.shape)
print("\nsource_dataset value counts:")
print(df["source_dataset"].value_counts())
print("\ndf.head():")
df.head()

Saved unified dataset to: /content/drive/MyDrive/bias_detection_project/data/processed/ingested_resumes.csv

df.shape: (26484, 13)

source_dataset value counts:
source_dataset
faircv           24000
kaggle_resume     2484
Name: count, dtype: int64

df.head():


,resume_id,raw_text,category_label,bio_text,name,gender_label,ethnicity_label,occupation_label,suitability_score,blind_score,biased_score_gender,biased_score_ethnicity,source_dataset
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,HR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,kaggle_resume
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...",HR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,kaggle_resume
2,33176873,HR DIRECTOR Summary Over 2...,HR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,kaggle_resume
3,27018550,HR SPECIALIST Summary Dedica...,HR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,kaggle_resume
4,17812897,HR MANAGER Skill Highlights ...,HR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,kaggle_resume


In [ ]:
%%writefile /content/drive/MyDrive/bias_detection_project/code/preprocessor.py
"""
preprocessor.py

Cleans and tokenizes free-text resume data, and converts FairCVdb's
structured records into free-text biography sentences so both data
sources share one text representation before scoring.
"""

from __future__ import annotations

import re
from typing import Dict, List, Optional

import pandas as pd


# Small built-in stopword list (no NLTK dependency).
STOPWORDS = {
    "a", "an", "the", "and", "or", "but", "if", "then", "so", "of", "to",
    "in", "on", "at", "for", "with", "as", "by", "from", "is", "are",
    "was", "were", "be", "been", "being", "this", "that", "these",
    "those", "it", "its", "he", "she", "they", "we", "you", "i", "his",
    "her", "their", "our", "your", "my", "will", "would", "can", "could",
    "should", "has", "have", "had", "do", "does", "did", "not", "no",
    "into", "about", "than", "which", "who", "whom", "such",
}

# Common resume boilerplate phrases to strip during cleaning.
BOILERPLATE_PHRASES = [
    "references available upon request",
    "references furnished upon request",
    "curriculum vitae",
    "to whom it may concern",
    "objective statement",
]

_TOKEN_RE = re.compile(r"[a-z0-9]+(?:-[a-z0-9]+)*")


class Preprocessor:
    """Cleans, tokenizes, and normalizes resume/biography text."""

    def clean(self, text: str) -> str:
        """Lowercase, strip punctuation (except intra-word hyphens),
        collapse whitespace, and remove common resume boilerplate.

        Args:
            text: Raw input text.

        Returns:
            Cleaned text. Returns an empty string for None/NaN/empty input.
        """
        if text is None or (isinstance(text, float) and pd.isna(text)):
            return ""

        cleaned = str(text).lower()

        for phrase in BOILERPLATE_PHRASES:
            cleaned = cleaned.replace(phrase, " ")

        # Keep letters, digits, whitespace, and intra-word hyphens.
        cleaned = re.sub(r"(?<=\w)-(?=\w)", "\x00HYPHEN\x00", cleaned)
        cleaned = re.sub(r"[^\w\s\x00]", " ", cleaned)
        cleaned = cleaned.replace("\x00HYPHEN\x00", "-")

        cleaned = re.sub(r"\s+", " ", cleaned).strip()
        return cleaned

    def tokenize(self, text: str) -> List[str]:
        """Tokenize text on word boundaries and remove stopwords.

        Args:
            text: Input text (ideally already cleaned).

        Returns:
            List of lowercase tokens with stopwords removed. Empty list
            for empty/None input.
        """
        if not text:
            return []

        tokens = _TOKEN_RE.findall(text.lower())
        return [t for t in tokens if t not in STOPWORDS]

    def _parse_faircv_raw_text(self, raw_text: str) -> Dict[str, str]:
        """Parse the 'Key: value; Key: value; ...' string produced by
        ResumeIngestor for FairCVdb rows into a field dict.

        Args:
            raw_text: The semicolon-delimited structured record string.

        Returns:
            Dict mapping field name -> raw string value. Missing/blank
            fields are simply absent from the dict.
        """
        fields: Dict[str, str] = {}
        if not raw_text or not isinstance(raw_text, str):
            return fields

        for chunk in raw_text.split(";"):
            chunk = chunk.strip()
            if not chunk or ":" not in chunk:
                continue
            key, _, value = chunk.partition(":")
            key = key.strip()
            value = value.strip()
            if key and value:
                fields[key] = value

        return fields

    def _describe_score(self, value_str: str, low: str, mid: str, high: str) -> Optional[str]:
        """Convert a 0-1 continuous score string into a qualitative phrase.

        Args:
            value_str: The score as a string (e.g. "0.82").
            low: Phrase to use for scores < 0.33.
            mid: Phrase to use for scores in [0.33, 0.66].
            high: Phrase to use for scores > 0.66.

        Returns:
            The qualitative phrase, or None if value_str isn't parseable.
        """
        try:
            value = float(value_str)
        except (ValueError, TypeError):
            return None

        if value < 0.33:
            return low
        if value <= 0.66:
            return mid
        return high

    def align_faircv_record(self, row: pd.Series) -> str:
        """Convert a FairCVdb structured record into a free-text biography
        sentence, matching the free-text shape of the Kaggle resumes.

        Args:
            row: A row (pandas Series) from the ingested DataFrame, whose
                `raw_text` field holds the semicolon-delimited structured
                record produced by ResumeIngestor for faircv rows.

        Returns:
            A short biography sentence. Missing fields are skipped
            gracefully rather than inserting "None" placeholders. Returns
            an empty string if no usable fields are found.
        """
        raw_text = row.get("raw_text", "") if hasattr(row, "get") else row["raw_text"]
        fields = self._parse_faircv_raw_text(raw_text)

        # No parseable fields at all -> nothing to describe.
        if not fields:
            return ""

        clauses: List[str] = []

        occupation = fields.get("Occupation")
        if occupation and occupation.lower() != "unknown":
            clauses.append(f"Candidate for a {occupation} position")
        else:
            clauses.append("Candidate")

        experience_desc = self._describe_score(
            fields.get("Experience", ""),
            low="limited professional experience",
            mid="a moderate amount of professional experience",
            high="extensive professional experience",
        )
        if experience_desc:
            clauses.append(f"with {experience_desc}")

        education_desc = self._describe_score(
            fields.get("Education", ""),
            low="a basic level of formal education",
            mid="an undergraduate level of education",
            high="a graduate level of education",
        )
        if education_desc:
            clauses.append(f"and {education_desc}")

        recommendation = fields.get("Recommendation")
        if recommendation is not None:
            rec_lower = recommendation.strip().lower()
            if rec_lower in {"1", "1.0", "true", "yes"}:
                clauses.append("holding a positive recommendation letter")
            elif rec_lower in {"0", "0.0", "false", "no"}:
                clauses.append("without a recommendation letter on file")

        availability_desc = self._describe_score(
            fields.get("Availability", ""),
            low="limited current availability",
            mid="moderate current availability",
            high="high current availability",
        )
        if availability_desc:
            clauses.append(f"and {availability_desc}")

        suitability_desc = self._describe_score(
            fields.get("Suitability", ""),
            low="a low overall suitability score",
            mid="a moderate overall suitability score",
            high="a strong overall suitability score",
        )
        if suitability_desc:
            clauses.append(f"reflecting {suitability_desc}")

        sentence = " ".join(clauses).strip()
        sentence = re.sub(r"\s+", " ", sentence)
        if not sentence.endswith("."):
            sentence += "."
        return sentence[0].upper() + sentence[1:] if sentence else sentence

    def process(self, df: pd.DataFrame) -> pd.DataFrame:
        """Apply cleaning, FairCVdb-to-text alignment, and tokenization to
        an ingested DataFrame.

        Args:
            df: DataFrame produced by ResumeIngestor.ingest_all(), containing
                at least `raw_text` and `source_dataset` columns.

        Returns:
            A copy of df with two new columns: `clean_text` and `tokens`.
        """
        result = df.copy()

        is_faircv = result["source_dataset"] == "faircv"

        # For faircv rows, first convert the structured record to a
        # free-text biography sentence; for all other rows, use raw_text
        # as-is. Then clean everything uniformly.
        pre_clean_text = result["raw_text"].copy()
        if is_faircv.any():
            pre_clean_text.loc[is_faircv] = result.loc[is_faircv].apply(
                self.align_faircv_record, axis=1
            )

        result["clean_text"] = pre_clean_text.apply(self.clean)
        result["tokens"] = result["clean_text"].apply(self.tokenize)

        return result

Overwriting /content/drive/MyDrive/bias_detection_project/code/preprocessor.py


In [ ]:
path = "/content/drive/MyDrive/bias_detection_project/code/preprocessor.py"

with open(path) as f:
    content = f.read()

print("File size (chars):", len(content))
print("Contains '_parse_faircv_raw_text':", "_parse_faircv_raw_text" in content)
print("Contains 'if not fields:' fix:", "if not fields:" in content)

File size (chars): 8569
Contains '_parse_faircv_raw_text': True
Contains 'if not fields:' fix: True


In [ ]:
%%writefile /content/drive/MyDrive/bias_detection_project/code/tests/test_preprocessor.py
"""
Tests for preprocessor.py: clean(), tokenize(), and align_faircv_record().
"""

import os
import sys

import pandas as pd
import pytest

sys.path.insert(
    0,
    os.path.join(os.path.dirname(__file__), ".."),
)

from preprocessor import Preprocessor  # noqa: E402


@pytest.fixture
def preprocessor() -> Preprocessor:
    """Provides a fresh Preprocessor instance for each test."""
    return Preprocessor()


class TestClean:
    """Tests for Preprocessor.clean()."""

    def test_removes_boilerplate_phrase(self, preprocessor: Preprocessor) -> None:
        text = "Experienced engineer. References available upon request."
        result = preprocessor.clean(text)
        assert "references available upon request" not in result

    def test_lowercases_and_strips_punctuation_but_keeps_hyphens(
        self, preprocessor: Preprocessor
    ) -> None:
        text = "Senior Full-Stack Engineer, Python/Java expert!!"
        result = preprocessor.clean(text)
        assert result == result.lower()
        assert "full-stack" in result
        assert "," not in result
        assert "!" not in result
        assert "/" not in result

    def test_collapses_whitespace(self, preprocessor: Preprocessor) -> None:
        text = "Too    many\n\nspaces\tand tabs"
        result = preprocessor.clean(text)
        assert "  " not in result

    def test_empty_and_none_input(self, preprocessor: Preprocessor) -> None:
        assert preprocessor.clean("") == ""
        assert preprocessor.clean(None) == ""


class TestTokenize:
    """Tests for Preprocessor.tokenize()."""

    def test_empty_string_returns_empty_list(self, preprocessor: Preprocessor) -> None:
        assert preprocessor.tokenize("") == []

    def test_none_input_returns_empty_list(self, preprocessor: Preprocessor) -> None:
        assert preprocessor.tokenize(None) == []  # type: ignore[arg-type]

    def test_removes_stopwords(self, preprocessor: Preprocessor) -> None:
        text = "the candidate is a strong engineer with the skills"
        tokens = preprocessor.tokenize(text)
        assert "the" not in tokens
        assert "is" not in tokens
        assert "a" not in tokens
        assert "with" not in tokens
        assert "candidate" in tokens
        assert "engineer" in tokens

    def test_preserves_hyphenated_words(self, preprocessor: Preprocessor) -> None:
        text = "full-stack developer"
        tokens = preprocessor.tokenize(text)
        assert "full-stack" in tokens


class TestAlignFaircvRecord:
    """Tests for Preprocessor.align_faircv_record()."""

    def test_full_record_produces_sentence(self, preprocessor: Preprocessor) -> None:
        raw_text = (
            "Occupation: nurse; Suitability: 0.8; Education: 0.7; "
            "Experience: 0.6; Recommendation: 1; Availability: 0.9; "
            "Language proficiency: [0.5, 0.6, 0.7]"
        )
        row = pd.Series({"raw_text": raw_text, "source_dataset": "faircv"})
        result = preprocessor.align_faircv_record(row)

        assert "nurse" in result
        assert result.endswith(".")
        assert "None" not in result

    def test_record_missing_several_fields(self, preprocessor: Preprocessor) -> None:
        raw_text = "Occupation: teacher; Suitability: 0.9"
        row = pd.Series({"raw_text": raw_text, "source_dataset": "faircv"})
        result = preprocessor.align_faircv_record(row)

        assert "teacher" in result
        assert "None" not in result
        assert "none" not in result.lower()
        assert len(result) > 0

    def test_completely_empty_record_returns_empty_string(
        self, preprocessor: Preprocessor
    ) -> None:
        row = pd.Series({"raw_text": "", "source_dataset": "faircv"})
        result = preprocessor.align_faircv_record(row)
        assert result == ""

    def test_malformed_raw_text_does_not_crash(self, preprocessor: Preprocessor) -> None:
        row = pd.Series({"raw_text": "not a valid key value format", "source_dataset": "faircv"})
        result = preprocessor.align_faircv_record(row)
        assert isinstance(result, str)

Overwriting /content/drive/MyDrive/bias_detection_project/code/tests/test_preprocessor.py


In [ ]:
!find /content/drive/MyDrive/bias_detection_project/code -name "__pycache__" -exec rm -rf {} + 2>/dev/null
!pip install -q pytest
!cd /content/drive/MyDrive/bias_detection_project/code && python -m pytest tests/ -v -p no:cacheprovider

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
rootdir: /content/drive/MyDrive/bias_detection_project/code
plugins: typeguard-4.6.0, langsmith-0.12.1, anyio-4.14.2
collected 12 items                                                             

tests/test_preprocessor.py::TestClean::test_removes_boilerplate_phrase PASSED [  8%]
tests/test_preprocessor.py::TestClean::test_lowercases_and_strips_punctuation_but_keeps_hyphens PASSED [ 16%]
tests/test_preprocessor.py::TestClean::test_collapses_whitespace PASSED  [ 25%]
tests/test_preprocessor.py::TestClean::test_empty_and_none_input PASSED  [ 33%]
tests/test_preprocessor.py::TestTokenize::test_empty_string_returns_empty_list PASSED [ 41%]
tests/test_preprocessor.py::TestTokenize::test_none_input_returns_empty_list PASSED [ 50%]
tests/test_preprocessor.py::TestTokenize::test_removes_stopwords PASSED  [ 58%]
tests/test_preprocess

In [ ]:
import os
import sys
import random

import pandas as pd

PROJECT_ROOT = "/content/drive/MyDrive/bias_detection_project"
CODE_DIR = os.path.join(PROJECT_ROOT, "code")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

sys.path.insert(0, CODE_DIR)

from preprocessor import Preprocessor  # noqa: E402

input_path = os.path.join(PROCESSED_DIR, "ingested_resumes.csv")
output_path = os.path.join(PROCESSED_DIR, "preprocessed_resumes.csv")

df = pd.read_csv(input_path)

preprocessor = Preprocessor()
processed_df = preprocessor.process(df)

processed_df.to_csv(output_path, index=False)
print(f"Saved preprocessed dataset to: {output_path}\n")

# --- Sanity check 1: average token count per source_dataset ---
processed_df["token_count"] = processed_df["tokens"].apply(len)
avg_tokens = processed_df.groupby("source_dataset")["token_count"].mean()
print("Average token count per source_dataset:")
print(avg_tokens)
print()

# --- Sanity check 2: 3 random before/after examples ---
sample_rows = processed_df.sample(
    n=min(3, len(processed_df)), random_state=random.randint(0, 10_000)
)

print("=== Random before/after examples ===")
for idx, row in sample_rows.iterrows():
    print(f"\n--- Row index {idx} | source_dataset={row['source_dataset']} ---")
    print("BEFORE (raw_text):")
    print(str(row["raw_text"])[:300])
    print("\nAFTER (clean_text):")
    print(str(row["clean_text"])[:300])
    print(f"\nTokens ({row['token_count']}):", row["tokens"][:20])

Saved preprocessed dataset to: /content/drive/MyDrive/bias_detection_project/data/processed/preprocessed_resumes.csv

Average token count per source_dataset:
source_dataset
faircv            21.786792
kaggle_resume    632.692029
Name: token_count, dtype: float64

=== Random before/after examples ===

--- Row index 10758 | source_dataset=faircv ---
BEFORE (raw_text):
Occupation: attorney; Suitability: 0.5; Education: 0.8; Experience: 0.6; Recommendation: 0.0; Availability: 0.8; Language proficiency: [np.float64(0.6), np.float64(0.2), np.float64(0.6)]

AFTER (clean_text):
candidate for a attorney position with a moderate amount of professional experience and a graduate level of education without a recommendation letter on file and high current availability reflecting a moderate overall suitability score

Tokens (22): ['candidate', 'attorney', 'position', 'moderate', 'amount', 'professional', 'experience', 'graduate', 'level', 'education', 'without', 'recommendation', 'letter', 'file', 'h

In [ ]:
import os
PROJECT_ROOT = "/content/drive/MyDrive/bias_detection_project/"
for sub in ["code", "tests", "data/processed"]:
    os.makedirs(os.path.join(PROJECT_ROOT, sub), exist_ok=True)

In [ ]:
%%writefile /content/drive/MyDrive/bias_detection_project/code/name_lists.py
"""
Name lists for counterfactual (name-swap) resume auditing.

Source
------
Bertrand, M., & Mullainathan, S. (2004). "Are Emily and Greg More Employable
Than Lakisha and Jamal? A Field Experiment on Labor Market Discrimination."
American Economic Review, 94(4), 991-1013.

IMPORTANT: the lists below are an ABBREVIATED subset of the names used in the
paper. A real submission should use the COMPLETE published list from the
paper's appendix, not this shortened one, and verify every entry against it
(the last-name lists in particular).

Two attribute dimensions
------------------------
1. Name-associated group  -> NAME_GROUPS   ("groupA", "groupB")
2. Gender-associated first names -> GENDER_FIRST_NAMES ("female", "male")

Group keys are deliberately neutral labels. Their provenance is kept in each
group's "b_and_m_label" field:
    groupA = names the paper classifies as White-sounding
    groupB = names the paper classifies as African-American-sounding

Within each group, first names are split by gender so that an intersectional
label such as "female_groupB" maps to exactly one pool of first names.
GENDER_FIRST_NAMES is derived from those same pools (pooled across groups) and
is available for gender-only swaps.

These are name-based PROXIES used to build counterfactuals. They are not
ground-truth demographic attributes of any real applicant.
"""

NAME_GROUPS = {
    "groupA": {
        "b_and_m_label": "White-sounding",
        "first_names": {
            "female": ["Allison", "Anne", "Carrie", "Emily", "Jill",
                       "Laurie", "Kristen", "Meredith", "Sarah"],
            "male": ["Brad", "Brendan", "Geoffrey", "Greg", "Brett",
                     "Jay", "Matthew", "Neil", "Todd"],
        },
        "last_names": ["Baker", "Kelly", "McCarthy", "Murphy", "Murray",
                       "O'Brien", "Ryan", "Sullivan", "Walsh"],
    },
    "groupB": {
        "b_and_m_label": "African-American-sounding",
        "first_names": {
            "female": ["Aisha", "Ebony", "Keisha", "Kenya", "Lakisha",
                       "Latonya", "Latoya", "Tamika", "Tanisha"],
            "male": ["Darnell", "Hakim", "Jermaine", "Kareem", "Jamal",
                     "Leroy", "Rasheed", "Tremayne", "Tyrone"],
        },
        "last_names": ["Jackson", "Jones", "Robinson", "Washington",
                       "Williams", "Jefferson", "Booker"],
    },
}

GENDER_GROUPS = ["female", "male"]

# Gender dimension: pooled first names per gender, derived from NAME_GROUPS.
GENDER_FIRST_NAMES = {
    g: sorted({n for grp in NAME_GROUPS.values() for n in grp["first_names"][g]})
    for g in GENDER_GROUPS
}

Writing /content/drive/MyDrive/bias_detection_project/code/name_lists.py


In [ ]:
%%writefile /content/drive/MyDrive/bias_detection_project/code/name_swap_auditor.py
"""
NameSwapAuditor: counterfactual resume generation via name substitution.

Each resume is duplicated once per intersectional group (gender x name group,
e.g. "female_groupB"), with only the name changed. Any difference in a model's
output between variants of the same resume can then be attributed to the name.
No real personal data is used, and resumes with no native demographic label
(e.g. the Kaggle 24-class set) receive a proxy label by construction.

FairCVdb note
-------------
For FairCVdb rows, this module creates an ADDITIONAL demographic label
(`intersectional_group`) alongside the dataset's own native gender/ethnicity
labels. Both are carried through side by side, so the FairnessAuditor module
can use either and compare them.

Interpretation note: the name-swap label describes the *variant*, not the
original applicant. It measures a model's sensitivity to names; it is not a
claim about the real candidate's demographics.
"""
import hashlib
import random
from typing import Optional

import pandas as pd

from name_lists import NAME_GROUPS, GENDER_GROUPS

PLACEHOLDER = "FULL_NAME_HERE"


class NameSwapAuditor:
    def __init__(self, seed: int = 42):
        self.seed = seed
        self.name_groups = list(NAME_GROUPS.keys())
        self.gender_groups = list(GENDER_GROUPS)
        # Order: gender-major, e.g. female_groupA, female_groupB, male_groupA, ...
        self.labels = [
            self.build_intersectional_label(n, g)
            for g in self.gender_groups
            for n in self.name_groups
        ]

    # ------------------------------------------------------------------ #
    def insert_placeholder(self, text: str) -> str:
        """Prepend PLACEHOLDER as an opening header line. Idempotent."""
        if text is None or (isinstance(text, float) and pd.isna(text)):
            text = ""
        if PLACEHOLDER in text:
            return text
        return f"{PLACEHOLDER}\n\n{text}" if text else PLACEHOLDER

    # ------------------------------------------------------------------ #
    def build_intersectional_label(self, name_group: str, gender_group: str) -> str:
        """Combine both dimensions, e.g. ("groupB", "female") -> "female_groupB"."""
        if name_group not in NAME_GROUPS:
            raise ValueError(f"Unknown name group: {name_group!r}")
        if gender_group not in GENDER_GROUPS:
            raise ValueError(f"Unknown gender group: {gender_group!r}")
        return f"{gender_group}_{name_group}"

    # ------------------------------------------------------------------ #
    def _pick_name(self, name_group: str, gender_group: str, key: str) -> str:
        """Deterministic name choice per (key, group, gender) so runs reproduce."""
        grp = NAME_GROUPS[name_group]
        rng = random.Random(f"{self.seed}|{key}|{name_group}|{gender_group}")
        first = rng.choice(grp["first_names"][gender_group])
        last = rng.choice(grp["last_names"])
        return f"{first} {last}"

    def generate_counterfactuals(
        self, resume_text: str, key: Optional[str] = None
    ) -> dict[str, str]:
        """
        Return {intersectional_label: resume_text_with_name} for every
        gender x name-group combination. Everything except the name is
        identical across variants.

        `key` seeds name selection (process_dataset passes the resume's id).
        If omitted it is derived from the resume text.
        """
        with_ph = self.insert_placeholder(resume_text)
        if key is None:
            key = hashlib.md5(with_ph.encode("utf-8")).hexdigest()[:12]
        out = {}
        for gender in self.gender_groups:
            for ng in self.name_groups:
                label = self.build_intersectional_label(ng, gender)
                name = self._pick_name(ng, gender, str(key))
                out[label] = with_ph.replace(PLACEHOLDER, name)
        return out

    # ------------------------------------------------------------------ #
    def process_dataset(
        self, df: pd.DataFrame, text_col: str = "clean_text"
    ) -> pd.DataFrame:
        """
        Long-format expansion: every resume x every intersectional group.

        Output columns (first): resume_id, original_id, intersectional_group,
        name_group, gender_group, name_used, counterfactual_text; followed by
        all original columns unchanged. If the input has a `resume_id` column
        it becomes `original_id`, and `resume_id` is replaced by a unique
        per-variant id ("<original_id>__<label>"). Otherwise the row position
        is used as original_id.

        FairCVdb rows keep their native gender/ethnicity columns, so both
        label sets are available downstream (see module docstring).
        """
        base = df.reset_index(drop=True).copy()
        if "resume_id" in base.columns:
            original_ids = base["resume_id"].astype(str)
        else:
            original_ids = pd.Series(base.index.astype(str), index=base.index)

        cf_list = [
            self.generate_counterfactuals(t, key=oid)
            for t, oid in zip(base[text_col].tolist(), original_ids.tolist())
        ]

        pieces = []
        for gender in self.gender_groups:
            for ng in self.name_groups:
                label = self.build_intersectional_label(ng, gender)
                piece = base.copy()
                piece["original_id"] = original_ids
                piece["resume_id"] = original_ids + "__" + label
                piece["intersectional_group"] = label
                piece["name_group"] = ng
                piece["gender_group"] = gender
                piece["counterfactual_text"] = [cf[label] for cf in cf_list]
                piece["name_used"] = [
                    cf[label].split("\n", 1)[0] if cf[label] else ""
                    for cf in cf_list
                ]
                pieces.append(piece)

        long_df = pd.concat(pieces).sort_index(kind="stable").reset_index(drop=True)

        front = ["resume_id", "original_id", "intersectional_group", "name_group",
                 "gender_group", "name_used", "counterfactual_text"]
        rest = [c for c in long_df.columns if c not in front]
        return long_df[front + rest]

Writing /content/drive/MyDrive/bias_detection_project/code/name_swap_auditor.py


In [ ]:
%%writefile /content/drive/MyDrive/bias_detection_project/tests/test_name_swap_auditor.py
import sys
from pathlib import Path

import pandas as pd
import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "code"))
from name_swap_auditor import NameSwapAuditor, PLACEHOLDER  # noqa: E402
from name_lists import NAME_GROUPS  # noqa: E402


@pytest.fixture
def auditor():
    return NameSwapAuditor(seed=0)


def n_labels(a):
    return len(a.labels)


def test_resume_without_existing_name(auditor):
    text = "experienced java developer with 5 years in backend systems"
    assert auditor.insert_placeholder(text).startswith(PLACEHOLDER)

    cfs = auditor.generate_counterfactuals(text)
    assert set(cfs) == {"female_groupA", "female_groupB",
                        "male_groupA", "male_groupB"}
    for label, out in cfs.items():
        assert PLACEHOLDER not in out
        name, body = out.split("\n\n", 1)
        assert body == text  # everything except the name is identical
        gender, group = label.split("_")
        first, last = name.split(" ", 1)
        assert first in NAME_GROUPS[group]["first_names"][gender]
        assert last in NAME_GROUPS[group]["last_names"]


def test_empty_string(auditor):
    assert auditor.insert_placeholder("") == PLACEHOLDER
    cfs = auditor.generate_counterfactuals("")
    assert len(cfs) == n_labels(auditor)
    assert all(v.strip() and PLACEHOLDER not in v for v in cfs.values())


def test_insert_placeholder_idempotent(auditor):
    once = auditor.insert_placeholder("some text")
    assert auditor.insert_placeholder(once) == once


def test_deterministic(auditor):
    t = "data analyst"
    assert auditor.generate_counterfactuals(t) == auditor.generate_counterfactuals(t)


def test_intersectional_label(auditor):
    assert auditor.build_intersectional_label("groupB", "female") == "female_groupB"
    with pytest.raises(ValueError):
        auditor.build_intersectional_label("groupZ", "female")
    with pytest.raises(ValueError):
        auditor.build_intersectional_label("groupA", "other")


def test_process_dataset_row_multiplication(auditor):
    df = pd.DataFrame({
        "resume_id": ["r1", "r2", "r3"],
        "clean_text": ["python developer", "", "nurse with icu experience"],
        "category_label": ["IT", "HEALTH", "HEALTH"],
        "suitability_score": [0.9, 0.1, 0.5],
        "source_dataset": ["kaggle", "faircvdb", "faircvdb"],
    })
    out = auditor.process_dataset(df)

    k = n_labels(auditor)
    assert len(out) == len(df) * k
    assert out["resume_id"].is_unique
    assert set(out["original_id"]) == {"r1", "r2", "r3"}
    assert (out.groupby("original_id")["intersectional_group"].nunique() == k).all()
    assert out["intersectional_group"].value_counts().nunique() == 1
    for col in ["resume_id", "original_id", "intersectional_group",
                "counterfactual_text", "category_label",
                "suitability_score", "source_dataset", "clean_text"]:
        assert col in out.columns
    # original columns carried through unchanged
    r1 = out[out["original_id"] == "r1"]
    assert (r1["category_label"] == "IT").all()
    assert (r1["clean_text"] == "python developer").all()


def test_process_dataset_without_resume_id(auditor):
    df = pd.DataFrame({"clean_text": ["a", "b"]})
    out = auditor.process_dataset(df)
    assert len(out) == 2 * n_labels(auditor)
    assert set(out["original_id"]) == {"0", "1"}

Writing /content/drive/MyDrive/bias_detection_project/tests/test_name_swap_auditor.py


In [ ]:
!pip -q install pytest
!cd /content/drive/MyDrive/bias_detection_project && python -m pytest -q tests/

.......                                                                  [100%]
7 passed in 0.68s


In [ ]:
import sys, importlib
import pandas as pd

PROJECT_ROOT = "/content/drive/MyDrive/bias_detection_project/"
sys.path.insert(0, PROJECT_ROOT + "code")
import name_lists, name_swap_auditor
importlib.reload(name_lists); importlib.reload(name_swap_auditor)
from name_swap_auditor import NameSwapAuditor

df = pd.read_csv(PROJECT_ROOT + "data/processed/preprocessed_resumes.csv")
auditor = NameSwapAuditor(seed=42)

cf = auditor.process_dataset(df)
out_path = PROJECT_ROOT + "data/processed/counterfactual_resumes.csv"
cf.to_csv(out_path, index=False)

k = len(auditor.labels)
print(f"Original rows:  {len(df):,}")
print(f"Groups:         {k}")
print(f"Expected rows:  {len(df) * k:,}")
print(f"Actual rows:    {len(cf):,}")
assert len(cf) == len(df) * k

print("\nintersectional_group counts:")
print(cf["intersectional_group"].value_counts())

Original rows:  26,484
Groups:         4
Expected rows:  105,936
Actual rows:    105,936

intersectional_group counts:
intersectional_group
female_groupA    26484
female_groupB    26484
male_groupA      26484
male_groupB      26484
Name: count, dtype: int64


-----------------------------------------------------------------------------------

In [ ]:
import sys, importlib, json, numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score

PROJECT_ROOT = "/content/drive/MyDrive/bias_detection_project/"
sys.path.insert(0, PROJECT_ROOT + "code")
import baseline_scorer; importlib.reload(baseline_scorer)
from baseline_scorer import BaselineScorer, ensure_split

RAW = PROJECT_ROOT + "data/raw/faircvdb/FairCVdb.npy"
PAPER_BERTIDRAAF_ACC = {"kaggle": None, "faircv": None}   # fractions, from the paper
BERT_LARGE_GAP_KAGGLE = 0.0322

df = pd.read_csv(PROJECT_ROOT + "data/processed/counterfactual_resumes.csv", low_memory=False)
src = df["source_dataset"].astype(str).str.lower()
kaggle = df[src.str.contains("kaggle")].copy()
fc = df[src.str.contains("faircv")].copy()

# ---- merge the real labels + native split back in (original_id = faircv_<split>_<row index>) ----
d = np.load(RAW, allow_pickle=True)
raw = d.item() if (d.dtype == object and d.shape == ()) else d
def labels(split):
    n = len(raw[f"Profiles {split}"])
    return pd.DataFrame({
        "original_id": [f"faircv_{split.lower()}_{i}" for i in range(n)],
        "score_blind": raw[f"Blind Labels {split}"],
        "score_biased_gender": raw[f"Biased Labels {split} (Gender)"],
        "score_biased_ethnicity": raw[f"Biased Labels {split} (Ethnicity)"],
        "split": "train" if split == "Train" else "test"})
lab = pd.concat([labels("Train"), labels("Test")], ignore_index=True)
fc = fc.drop(columns=["suitability_score", "split"], errors="ignore").merge(
    lab, on="original_id", how="left", validate="m:1")
assert fc["score_blind"].notna().all(), "original_id values did not match the npy rows"

# carve a validation set out of the native train profiles (fixed seed)
tr_ids = fc.loc[fc.split == "train", "original_id"].unique()
val_ids = np.random.RandomState(42).choice(tr_ids, size=1920, replace=False)
fc.loc[fc.original_id.isin(val_ids), "split"] = "val"
print(fc.groupby("split")["original_id"].nunique())          # expect 17280 / 1920 / 4800

TARGETS = ["score_blind", "score_biased_gender", "score_biased_ethnicity"]
# FIXED binary cut-off per target = median of that label over the native TRAIN profiles.
THR = {t: float(fc.loc[fc.split == "train", t].median()) for t in TARGETS}
print("thresholds:", THR)
json.dump(THR, open(PROJECT_ROOT + "data/processed/faircv_thresholds.json", "w"))

def verdict(name, acc):
    ref = PAPER_BERTIDRAAF_ACC.get(name)
    if ref is None:
        print("  (fill in PAPER_BERTIDRAAF_ACC to enable the sanity check)")
    elif acc > ref:
        print(f"  FLAG: tf-idf ({acc:.4f}) beats BERTidRAAF's reported {ref:.4f}. Investigate.")
    elif name == "kaggle" and acc > ref - BERT_LARGE_GAP_KAGGLE:
        print(f"  WARNING: above the paper's BERT-large ({ref - BERT_LARGE_GAP_KAGGLE:.4f}).")
    else:
        print("  OK: below the paper's reported numbers.")
    if acc >= 0.99:
        print("  WARNING: near-perfect accuracy, check for leakage.")

results = []

# ---- Kaggle (unchanged) ----
kaggle, created_k = ensure_split(kaggle, kaggle["category_label"])
tr, te = kaggle[kaggle.split == "train"], kaggle[kaggle.split == "test"]
assert not set(tr.original_id) & set(te.original_id)
s = BaselineScorer(); s.fit(tr)
out = s.predict_kaggle(te); out.insert(0, "dataset", "kaggle"); out.insert(1, "target", "category_label")
results.append(out)
acc = accuracy_score(out.true_label.astype(str), out.predicted_label.astype(str))
print(f"\nkaggle: train={len(tr)} test={len(te)} accuracy={acc:.4f}"); verdict("kaggle", acc)

# ---- FairCV: one run per target ----
for t in TARGETS:
    d_ = fc.assign(suitability_score=fc[t])      # BaselineScorer reads this column
    tr, te = d_[d_.split == "train"], d_[d_.split == "test"]
    assert not set(tr.original_id) & set(te.original_id)
    s = BaselineScorer(faircv_threshold=THR[t]); s.fit(tr)
    out = s.predict_faircv(te)
    out.insert(0, "dataset", "faircv"); out.insert(1, "target", t)
    results.append(out)

    y, yh = out.true_label.astype(int), out.predicted_label.astype(int)
    acc, auc = accuracy_score(y, yh), roc_auc_score(y, out.prob_positive)
    print(f"\nfaircv [{t}]: train={len(tr)} test={len(te)} thr={THR[t]:.4f} "
          f"pos rate={y.mean():.3f} accuracy={acc:.4f} AUC={auc:.4f}")
    verdict("faircv", acc)

    # name sensitivity: same resume, only the name varies
    p = out.prob_positive.clip(1e-9, 1 - 1e-9)
    lg = pd.Series(np.log(p / (1 - p)).to_numpy(), index=out.index)
    g = lg.groupby(out.original_id)
    print(f"  name-swap log-odds spread: mean={(g.max() - g.min()).mean():.4f} "
          f"max={(g.max() - g.min()).max():.4f} | flip rate="
          f"{out.groupby('original_id').predicted_label.nunique().gt(1).mean():.4%}")
    print("  predicted positive rate by name group:",
          out.groupby("intersectional_group").predicted_label.apply(lambda x: x.astype(int).mean()).round(4).to_dict())

pd.concat(results, ignore_index=True).to_csv(
    PROJECT_ROOT + "data/processed/baseline_predictions.csv", index=False)
pd.concat([kaggle[["resume_id", "split"]], fc[["resume_id", "split"]]]).to_csv(
    PROJECT_ROOT + "data/processed/splits.csv", index=False)      # reuse in ImprovedScorer

split
test      4800
train    17280
val       1920
Name: original_id, dtype: int64
thresholds: {'score_blind': 0.41373492308688775, 'score_biased_gender': 0.3657409924542915, 'score_biased_ethnicity': 0.41459000863786377}


/content/drive/MyDrive/bias_detection_project/code/baseline_scorer.py:98: UserWarning: No 'split' column found: creating a stratified 80/10/10 split grouped by 'original_id' (seed=42). Reuse this same split in ImprovedScorer so the comparison is consistent.
  warnings.warn(



kaggle: train=7948 test=996 accuracy=0.6386
  (fill in PAPER_BERTIDRAAF_ACC to enable the sanity check)

faircv [score_blind]: train=69120 test=19200 thr=0.4137 pos rate=0.492 accuracy=0.7990 AUC=0.8782
  (fill in PAPER_BERTIDRAAF_ACC to enable the sanity check)
  name-swap log-odds spread: mean=0.3040 max=0.8126 | flip rate=1.2083%
  predicted positive rate by name group: {'female_groupA': 0.5448, 'female_groupB': 0.5435, 'male_groupA': 0.5448, 'male_groupB': 0.5446}

faircv [score_biased_gender]: train=69120 test=19200 thr=0.3657 pos rate=0.495 accuracy=0.7799 AUC=0.8546
  (fill in PAPER_BERTIDRAAF_ACC to enable the sanity check)
  name-swap log-odds spread: mean=0.3017 max=0.8190 | flip rate=3.3333%
  predicted positive rate by name group: {'female_groupA': 0.539, 'female_groupB': 0.5375, 'male_groupA': 0.5312, 'male_groupB': 0.5425}

faircv [score_biased_ethnicity]: train=69120 test=19200 thr=0.4146 pos rate=0.489 accuracy=0.7326 AUC=0.8121
  (fill in PAPER_BERTIDRAAF_ACC to enabl

-------------------------------------------------------------------------------------------------


In [ ]:
# ===== Colab cell: BERTidRAAF-lite CONTROL (training-only IDF weights) vs IMPROVED (IBS weights), matched pair =====
# Mirrors the improved cell: same data / splits / FairCV targets+thresholds / scrubbing / config (joint Kaggle+FairCV,
# fine_tune=True, epochs=6, 20 min budget, no cut-off calibration). TEST split only. Each model gets its OWN engine.
# Never overwrites improved_predictions.csv (the locked file). The matched-run improved model goes to
# improved_predictions_matched*.csv; set RETRAIN_IMPROVED=False to compare against the locked file only.
import os, sys, time, gc, importlib
import numpy as np, pandas as pd, torch

PROJECT_ROOT = "/content/drive/MyDrive/bias_detection_project/"
sys.path.insert(0, PROJECT_ROOT + "code")
import baseline_scorer, ibs_calculator, embedding_engine, comparison, improved_scorer, data_prep, bertidraaf_lite_scorer
for m in (baseline_scorer, ibs_calculator, embedding_engine, comparison, improved_scorer, data_prep,
          bertidraaf_lite_scorer):
    importlib.reload(m)
from embedding_engine import EmbeddingEngine
from bertidraaf_lite_scorer import build_matched_pair, BERTidRAAFLiteScorer, BERTidRAAFPaperScorer
from data_prep import load_prepared

PROC = PROJECT_ROOT + "data/processed/"
TARGETS = ["score_blind", "score_biased_gender", "score_biased_ethnicity"]
SEEDS = [42]                  # >1 seed -> one file per seed (_seed{n}) for BOTH models
RETRAIN_IMPROVED = True       # True: improved is retrained in the same session/config as the control
IBS_WEIGHT_MODE = "parity"    # = the improved cell's weight_mode
CONTROL_VARIANT = "lite"      # "lite": agreed token-median OOV (fair control) | "paper": paper-literal max-IDF OOV
USE_V2_OPTIONS = False        # True: label_smoothing + swa_last_epochs for BOTH models (needs improved_scorer v2)
V2_OPTIONS = dict(label_smoothing=0.1, swa_last_epochs=3) if USE_V2_OPTIONS else {}
CONTROL_CLS = {"lite": BERTidRAAFLiteScorer, "paper": BERTidRAAFPaperScorer}[CONTROL_VARIANT]

OUT_CONTROL = PROC + ("bertidraaf_lite_predictions.csv" if CONTROL_VARIANT == "lite" else "bertidraaf_paper_predictions.csv")
OUT_IMPROVED = PROC + "improved_predictions_matched.csv"
IMPROVED_LOCKED = PROC + "improved_predictions.csv"          # read-only here
def seed_path(base, seed): return base if len(SEEDS) == 1 else base.replace(".csv", f"_seed{seed}.csv")

# ---- same data preparation as the improved cell (incl. FairCV suitability-phrase scrubbing) ----
all_df, THR = load_prepared(PROJECT_ROOT, apply_scrub=True)
train, val, test = (all_df[all_df.split == s] for s in ("train", "val", "test"))
src = lambda d, k: d["source_dataset"].astype(str).str.lower().str.contains(k, regex=False)
tdf_k, tdf_f = test[src(test, "kaggle")], test[src(test, "faircv")]
print(f"test rows: kaggle={len(tdf_k)} faircv={len(tdf_f)} | train={len(train)} val={len(val)} | THR={THR}")

# ---- column contract = the improved predictions file (checked BEFORE the long training) ----
REF_COLS = list(pd.read_csv(IMPROVED_LOCKED, nrows=0).columns)
EXPECTED = {"dataset", "target", "resume_id", "original_id", "intersectional_group", "true_label",
            "predicted_label", "confidence", "prob_positive", "fusion_gate"}
if set(REF_COLS) != EXPECTED:
    print(f"WARNING: {IMPROVED_LOCKED} columns differ from what the scorers emit. "
          f"only in file: {sorted(set(REF_COLS) - EXPECTED)} | only in scorer output: {sorted(EXPECTED - set(REF_COLS))}")

def assemble(scorer, path):
    """Long-format CSV exactly like the improved cell: Kaggle rows, then FairCV rows, 'dataset' inserted first."""
    pk = scorer.predict_kaggle(tdf_k); pk.insert(0, "dataset", "kaggle")
    pf = scorer.predict_faircv(tdf_f); pf.insert(0, "dataset", "faircv")
    out = pd.concat([pk, pf], ignore_index=True)
    if set(out.columns) != set(REF_COLS):
        out.to_csv(path.replace(".csv", "_UNALIGNED.csv"), index=False)      # keep the (expensive) result
        raise RuntimeError(f"Output columns differ from {IMPROVED_LOCKED}: only in output "
                           f"{sorted(set(out.columns) - set(REF_COLS))}, only in file {sorted(set(REF_COLS) - set(out.columns))}")
    out = out[REF_COLS]
    out.to_csv(path, index=False)
    return out

# ---- train the matched pair, one model at a time (free GPU memory in between) ----
preds = {}                                                   # (name, seed) -> predictions DataFrame
for seed in SEEDS:
    common = dict(epochs=6, fine_tune=True, faircv_targets=TARGETS, faircv_thresholds=THR,
                  time_budget_minutes=20, seed=seed, **V2_OPTIONS)   # the improved cell's kwargs (+ optional v2 options)
    pair = dict(zip(("improved", "control"),
                    build_matched_pair(common, IBS_WEIGHT_MODE, engine_factory=lambda: EmbeddingEngine(),
                                       control_cls=CONTROL_CLS)))
    for name in ("improved", "control"):
        sc = pair.pop(name)
        if name == "improved" and not RETRAIN_IMPROVED:
            del sc; continue
        t0 = time.time()
        sc.fit(train, val_df=val)                            # no calibrate_decision_thresholds (improved cell has none)
        out_path = seed_path(OUT_IMPROVED if name == "improved" else OUT_CONTROL, seed)
        preds[(name, seed)] = assemble(sc, out_path)
        print(f"[seed {seed}] {name}: fit+predict {(time.time() - t0) / 60:.1f} min -> {out_path}")
        del sc; gc.collect(); torch.cuda.empty_cache()
    pair.clear()

# ---- reference predictions ----
bp_k = pd.read_csv(PROC + "baseline_predictions.csv", low_memory=False, dtype={"resume_id": str, "original_id": str})
bp_f = pd.read_csv(PROC + "baseline_predictions_scrubbed.csv", low_memory=False, dtype={"resume_id": str, "original_id": str})    # scrubbed baseline, same split
bk, bf = bp_k[bp_k.dataset == "kaggle"], bp_f[bp_f.dataset == "faircv"]
imp_locked = pd.read_csv(IMPROVED_LOCKED, low_memory=False, dtype={"resume_id": str, "original_id": str})

# ---- checks: row count + (dataset, target, resume_id) key set ----
def keys(d):
    t = d["target"].fillna("").astype(str) if "target" in d else pd.Series("", index=d.index)
    return set(zip(d["dataset"].astype(str), t, d["resume_id"].astype(str).str.replace(r"\.0$", "", regex=True)))
refs = {"baseline_predictions.csv": bp_k, "improved_predictions.csv": imp_locked}
ok_all = True
for (name, seed), df in preds.items():
    if name != "control":
        continue
    for rname, ref in refs.items():
        same_n, same_k = len(df) == len(ref), keys(df) == keys(ref)
        ok_all &= same_n and same_k
        print(f"CHECK control[seed {seed}] vs {rname}: rows {len(df)} vs {len(ref)} -> {'PASS' if same_n else 'FAIL'} | "
              f"key set -> {'PASS' if same_k else 'FAIL'} (only-in-control {len(keys(df) - keys(ref))}, "
              f"only-in-ref {len(keys(ref) - keys(df))})")
    if ("improved", seed) in preds:
        m = preds[("improved", seed)]
        same_n, same_k = len(df) == len(m), keys(df) == keys(m)
        ok_all &= same_n and same_k
        print(f"CHECK control[seed {seed}] vs matched improved: rows {'PASS' if same_n else 'FAIL'} | keys {'PASS' if same_k else 'FAIL'}")
if "baseline_predictions.csv" in refs and len(bp_k) != len(bk) + len(bf):
    print("NOTE: baseline_predictions.csv has a different row count than kaggle(baseline)+faircv(scrubbed baseline); "
          "a FAIL against it for FairCV rows can come from that file rather than from the control.")
print("ALL CHECKS PASSED" if ok_all else "SOME CHECKS FAILED - see above")

# ---- accuracy / selection-rate table (test split) ----
lab = lambda s: s.astype(str).str.replace(r"\.0$", "", regex=True)
def kaggle_acc(o): return float((lab(o.true_label) == lab(o.predicted_label)).mean())
def f_acc(o):
    o = o[o.true_label.notna()]
    return float((lab(o.true_label) == lab(o.predicted_label)).mean())
def f_sel(o): return float((lab(o.predicted_label) == "1").mean())
def table(models):
    rows = {"kaggle accuracy": {n: kaggle_acc(d[d.dataset == "kaggle"]) for n, d in models.items()}}
    for t in TARGETS:
        sub = {n: d[(d.dataset == "faircv") & (d.target == t)] for n, d in models.items()}
        rows[f"{t} accuracy"] = {n: f_acc(s) for n, s in sub.items()}
        rows[f"{t} selection rate"] = {n: f_sel(s) for n, s in sub.items()}
    return pd.DataFrame(rows).T

tabs = []
for seed in SEEDS:
    models = {"baseline": pd.concat([bk, bf], ignore_index=True), f"control (IDF, {CONTROL_VARIANT})": preds[("control", seed)]}
    if ("improved", seed) in preds:
        models["improved (IBS, matched)"] = preds[("improved", seed)]
    models["improved (locked file)"] = imp_locked
    tb = table(models); tabs.append(tb)
    print(f"\n=== test split, seed {seed} (scrubbed text for all models) ===")
    print(tb.round(4).to_string())
if len(tabs) > 1:
    print("\n=== mean over seeds ===")
    print(pd.concat(tabs).groupby(level=0, sort=False).mean().round(4).to_string())
print("\nNOTE: 'improved (locked file)' Kaggle rows come from improved_predictions_kaggle_locked.csv (an earlier run); "
      "compare control vs 'improved (IBS, matched)' for the like-for-like IBS-vs-IDF effect.")

test rows: kaggle=996 faircv=19200 | train=77068 val=8672 | THR={'score_blind': 0.41373492308688775, 'score_biased_gender': 0.3657409924542915, 'score_biased_ethnicity': 0.41459000863786377}


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

... (3 outputs left out)

EmbeddingEngine: distilbert-base-uncased on cuda (hidden=768, max_length=512, fp16=True)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


EmbeddingEngine: distilbert-base-uncased on cuda (hidden=768, max_length=512, fp16=True)
       attribute                                                                                                                       improved                                                                                                                        control           status
  aux_reg_weight                                                                                                                            1.0                                                                                                                            1.0             same
     backbone_lr                                                                                                                          3e-05                                                                                                                          3e-05             same
      chunk_long                               

In [ ]:
# ===== Colab cell: FairCV = the OLD joint section ONLY (Kaggle+FairCV trained together, default config) =====
# Kaggle predictions are loaded from the locked file and never retrained.
# If the joint-section predictions (`joint_test`) are still in this Colab session, they are REUSED (seconds);
# otherwise the joint model is trained once (~9 min on a T4). No ensemble, no cut-off calibration.
import os, sys, time, shutil, importlib
import numpy as np, pandas as pd, torch

PROJECT_ROOT = "/content/drive/MyDrive/bias_detection_project/"
sys.path.insert(0, PROJECT_ROOT + "code")
import baseline_scorer, ibs_calculator, embedding_engine, comparison, improved_scorer, data_prep
for m in (baseline_scorer, ibs_calculator, embedding_engine, comparison, improved_scorer, data_prep):
    importlib.reload(m)
from improved_scorer import ImprovedScorer
from comparison import faircv_report, compare, paired_delta, verdict, placement
from data_prep import load_prepared

PROC = PROJECT_ROOT + "data/processed/"
TARGETS = ["score_blind", "score_biased_gender", "score_biased_ethnicity"]
PAPER_ACC = 0.8517                      # description only
SEED = 42

all_df, THR = load_prepared(PROJECT_ROOT, apply_scrub=True)
train, val, test = (all_df[all_df.split == s] for s in ("train", "val", "test"))
is_f = lambda d: d["source_dataset"].astype(str).str.lower().str.contains("faircv")
vdf_all, tdf_all = val[is_f(val)], test[is_f(test)]

# ---- Kaggle: locked, unchanged ----
LOCK = PROC + "improved_predictions_kaggle_locked.csv"
if not os.path.exists(LOCK):
    prev = pd.read_csv(PROC + "improved_predictions.csv", low_memory=False)
    prev[prev.dataset == "kaggle"].to_csv(LOCK, index=False)
    shutil.copy(PROC + "improved_predictions.csv", PROC + "improved_predictions_run_backup.csv")
pk = pd.read_csv(LOCK, low_memory=False)
print(f"Kaggle predictions loaded unchanged (n={len(pk)})")

bp_f = pd.read_csv(PROC + "baseline_predictions_scrubbed.csv", low_memory=False)   # scrubbed baseline, same split
bp_k = pd.read_csv(PROC + "baseline_predictions.csv", low_memory=False)
bk, bf = bp_k[bp_k.dataset == "kaggle"], bp_f[bp_f.dataset == "faircv"]
def kaggle_acc(o): return {"accuracy": float((o.true_label.astype(str) == o.predicted_label.astype(str)).mean())}

# ---- FairCV: the old joint section (default config, 0.5 cut-off) ----
if "joint_test" in globals() and len(joint_test) and len(joint_test[0]) == len(tdf_all) * len(TARGETS):
    print("Reusing the joint-section predictions already in memory (no retraining).")
    pj = joint_test[0].copy()
else:
    t0 = time.time()
    js = ImprovedScorer(epochs=6, fine_tune=True, weight_mode="parity", faircv_targets=TARGETS,
                        faircv_thresholds=THR, time_budget_minutes=20, seed=SEED)
    js.fit(train, val_df=val)                          # Kaggle + FairCV rows together, as in the old cell
    pj = js.predict_faircv(tdf_all)
    print(f"Joint section trained in {(time.time() - t0) / 60:.1f} min")
    del js; torch.cuda.empty_cache()
pj = pj.copy(); pj.insert(0, "dataset", "faircv")
pd.concat([pk, pj], ignore_index=True).to_csv(PROC + "improved_predictions.csv", index=False)

# ---- report ----
base_res, impr_res = {"kaggle": kaggle_acc(bk)}, {"kaggle": kaggle_acc(pk)}
for t in TARGETS:
    base_res[t], impr_res[t] = faircv_report(bf[bf.target == t]), faircv_report(pj[pj.target == t])
    r = impr_res[t]
    print(f"faircv [{t}]: acc={r['accuracy']:.4f} AUC={r['auc']:.4f} | spread mean={r['spread_mean']:.4f} "
          f"max={r['spread_max']:.4f} | flip={r['flip_rate']:.4%} | pos-rate by group {r['pos_rate_by_group']}")
table = compare(base_res, impr_res)
print("\n=== Baseline vs Improved (FairCV: old joint section) ===")
print(table.round(4).to_string(index=False))
table.to_csv(PROC + "comparison_baseline_vs_improved_faircv_joint_only.csv", index=False)

dk, dfc = paired_delta(bk, pk), paired_delta(bf, pj, key=("resume_id", "target"))
print("\n" + "=" * 78 + "\nFINAL ACCURACY STATEMENT (test split; scrubbed text for both models)\n" + "=" * 78)
print(f"Analysis 1 - Kaggle (UNCHANGED): baseline {dk['baseline']:.4f} | improved {dk['improved']:.4f} | "
      f"delta {dk['delta']:+.4f} (SE {dk['se']:.4f}) -> {verdict(dk)}")
print(f"Analysis 2 - FairCVdb pooled: baseline {dfc['baseline']:.4f} | improved {dfc['improved']:.4f} | "
      f"delta {dfc['delta']:+.4f} (SE {dfc['se']:.4f}) -> {verdict(dfc)}")
for t in TARGETS:
    d_t = paired_delta(bf[bf.target == t], pj[pj.target == t])
    print(f"     {t:24s} baseline {d_t['baseline']:.4f} | improved {d_t['improved']:.4f} | "
          f"delta {d_t['delta']:+.4f} (SE {d_t['se']:.4f})")
print("NOTE: SE covers test-set sampling only. Earlier runs of this same joint config scored 0.7717, 0.7647 and "
      "0.7715 pooled, so a gain below ~0.5 points should be described as parity, not as an improvement.")
print(f"Placement: Kaggle {placement(dk, PAPER_ACC)}\n           FairCV {placement(dfc, PAPER_ACC)}")
beaten = [n for n, d in (("Kaggle", dk), ("FairCVdb", dfc)) if d["delta"] <= 0]
print("RESULT: improved model is more accurate than the baseline on BOTH analyses (FairCV margin is small)." if not beaten
      else f"RESULT: improved model does NOT beat the baseline on: {', '.join(beaten)}. Reported as measured.")

Kaggle predictions loaded unchanged (n=996)
IBS lookup: 4 groups, 18048 terms | IBS<1: 53 terms, IBS==0: 20 terms, mean IBS 0.997
Training rows: Kaggle=7948  score_blind=69120  score_biased_gender=69120  score_biased_ethnicity=69120  device=cuda
FairCV thresholds: {'score_blind': 0.4137, 'score_biased_gender': 0.3657, 'score_biased_ethnicity': 0.4146}


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


EmbeddingEngine: distilbert-base-uncased on cuda (hidden=768, max_length=512, fp16=True)
Fine-tuning backbone (lr 3e-05) + gate/heads (lr 0.001): 665 steps/epoch x 6 epochs
Epoch 1/6 | loss 0.7714 | kaggle CE 1.8635 | faircv BCE 0.5186 | mean gate 0.451 | val kaggle acc 0.8226 | val faircv acc 0.7694 | 1.6 min
Epoch 2/6 | loss 0.5139 | kaggle CE 0.6701 | faircv BCE 0.4777 | mean gate 0.621 | val kaggle acc 0.8155 | val faircv acc 0.7818 | 1.6 min
Epoch 3/6 | loss 0.4641 | kaggle CE 0.4337 | faircv BCE 0.4711 | mean gate 0.658 | val kaggle acc 0.8427 | val faircv acc 0.7824 | 1.6 min
Epoch 4/6 | loss 0.4283 | kaggle CE 0.2669 | faircv BCE 0.4657 | mean gate 0.667 | val kaggle acc 0.8558 | val faircv acc 0.7759 | 1.5 min
Epoch 5/6 | loss 0.4073 | kaggle CE 0.1680 | faircv BCE 0.4626 | mean gate 0.667 | val kaggle acc 0.8528 | val faircv acc 0.7839 | 1.5 min
Epoch 6/6 | loss 0.3900 | kaggle CE 0.0874 | faircv BCE 0.4601 | mean gate 0.666 | val kaggle acc 0.8558 | val faircv acc 0.7837 | 1

------------------------------------------------------------------------------------------------------------------

In [ ]:
# ===== Colab cell: FairnessAuditor - three-model ladder =====
import os, sys
import pandas as pd

PROJECT_ROOT = "/content/drive/MyDrive/bias_detection_project/"
sys.path.insert(0, os.path.join(PROJECT_ROOT, "code"))
import importlib, fairness_auditor
importlib.reload(fairness_auditor)
from fairness_auditor import FairnessAuditor

PROC = os.path.join(PROJECT_ROOT, "data", "processed")
REPORTS = os.path.join(PROJECT_ROOT, "reports")
os.makedirs(REPORTS, exist_ok=True)

CONF = 0.90
GROUP_COLS = ["intersectional_group", "gender_label", "ethnicity_label"]
LADDER = ["baseline", "bertidraaf_lite", "improved"]

# ---- 1. load the four prediction files --------------------------------------
base_kaggle = pd.read_csv(os.path.join(PROC, "baseline_predictions.csv"))
base_faircv = pd.read_csv(os.path.join(PROC, "baseline_predictions_scrubbed.csv"))
lite = pd.read_csv(os.path.join(PROC, "bertidraaf_lite_predictions.csv"))
improved = pd.read_csv(os.path.join(PROC, "improved_predictions.csv"))

baseline = pd.concat([base_kaggle[base_kaggle["dataset"] == "kaggle"],
                      base_faircv[base_faircv["dataset"] == "faircv"]], ignore_index=True)
model_dfs = {"baseline": baseline, "bertidraaf_lite": lite, "improved": improved}

# ---- 2. same test rows for every model? (raises if not) ------------------------
aud = FairnessAuditor(n_iterations=1000, confidence=CONF, seed=42, cluster_col="original_id")
aud.validate_same_test_rows(model_dfs)

# ---- 3. native labels: original profile or swapped name? ------------------------
cf = pd.read_csv(os.path.join(PROC, "counterfactual_resumes.csv"))
native_cols = [c for c in ("resume_id", "gender_label", "ethnicity_label") if c in cf.columns]
native = cf[native_cols].copy()
print("\nNative label columns found:", native_cols)
info = aud.inspect_native_labels(improved, native)
print(info.to_string(index=False))
for r in info.itertuples():
    print(f"  -> {r.column}: {r.interpretation}")

# ---- 4. compare + pairwise ----------------------------------------------------------
comparison = aud.compare(model_dfs, GROUP_COLS, native_labels_df=native)
pairs = [("bertidraaf_lite", "improved"), ("baseline", "bertidraaf_lite"), ("baseline", "improved")]
paired = aud.pairwise_comparison(model_dfs, pairs, GROUP_COLS, native_labels_df=native,
                                 comparison_df=comparison)

comparison.to_csv(os.path.join(REPORTS, "fairness_comparison.csv"), index=False)
paired.to_csv(os.path.join(REPORTS, "fairness_paired_differences.csv"), index=False)

with pd.option_context("display.width", 250, "display.max_columns", None,
                       "display.max_rows", None, "display.float_format", "{:.4f}".format):
    print("\n=== fairness_comparison ===")
    print(comparison.to_string(index=False))
    print("\n=== fairness_paired_differences ===")
    print(paired.to_string(index=False))

# ---- 5. plain-language sentences, grouped by pair -----------------------------------------
HEADINGS = {
    ("bertidraaf_lite", "improved"): "Effect of IBS (the key comparison): bertidraaf_lite -> improved",
    ("baseline", "bertidraaf_lite"): "Effect of the BERT upgrade alone: baseline -> bertidraaf_lite",
    ("baseline", "improved"): "Total effect: baseline -> improved",
}
BASIS_NAME = {"accuracy": "accuracy ratio", "selection_rate": "selection-rate ratio"}
pct = f"{int(round(CONF * 100))}%"
yn = lambda b: "yes" if b else "no"

def sentence(r):
    where = f"[{r.dataset} / {r.target} / {r.group_attribute}]"
    f_from, f_to = f"{r.fairness_from:.3f}", f"{r.fairness_to:.3f}"
    ci = f"diff {r.fairness_diff:+.3f}, {pct} paired CI [{r.fairness_diff_low:+.3f}, {r.fairness_diff_high:+.3f}]"
    if r.direction == "higher":
        core = (f"{r.model_to} is HIGHER than {r.model_from} on the {BASIS_NAME[r.metric_basis]} "
                f"({f_to} vs {f_from}; {ci}).")
    elif r.direction == "lower":
        core = (f"{r.model_to} is NOT fairer: its {BASIS_NAME[r.metric_basis]} is LOWER than "
                f"{r.model_from}'s ({f_to} vs {f_from}; {ci}).")
    else:
        core = (f"{r.model_to} is EQUAL to {r.model_from} on the {BASIS_NAME[r.metric_basis]}: "
                f"no difference distinguishable from 0 (point estimates {f_to} vs {f_from}; {ci}).")
    return (f"{where} {core} Significant per is_significant_gap (non-overlap): "
            f"{yn(r.significant_non_overlap)}; per paired interval: {yn(r.significant_paired)}.")

ds_order = {"kaggle": 0, "faircv": 1}
for pair in pairs:
    sub = paired[(paired.model_from == pair[0]) & (paired.model_to == pair[1])]
    if sub.empty:
        continue
    print("\n" + "=" * 100 + f"\n{HEADINGS[pair]}\n" + "=" * 100)
    sub = sub.assign(_d=sub.dataset.map(ds_order)).sort_values(["_d", "target", "group_attribute"])
    for r in sub.itertuples():
        print("- " + sentence(r))

# ---- 6. ladder summary -----------------------------------------------------------------------
ladder = (comparison.pivot_table(index=["dataset", "target", "group_attribute"],
                                 columns="algorithm", values="fairness_measure")
          .reindex(columns=[m for m in LADDER if m in comparison.algorithm.unique()]))
print("\n=== Ladder summary: fairness (baseline / bertidraaf_lite / improved) ===")
with pd.option_context("display.width", 200, "display.float_format", "{:.3f}".format):
    print(ladder.to_string())

print("\nNote: CI covers test-set sampling only; earlier runs of this joint config varied by "
      "~0.5 points in accuracy. The bertidraaf_lite and improved models were each trained once; "
      "Kaggle improved predictions come from a locked earlier run.")

OK: all 3 models share the same 14,649 (dataset, target, original_id) combinations and row counts.


/tmp/ipykernel_40112/618378805.py:34: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  cf = pd.read_csv(os.path.join(PROC, "counterfactual_resumes.csv"))



Native label columns found: ['resume_id', 'gender_label', 'ethnicity_label']
         column  share_missing  n_distinct  share_profiles_varying                                                             interpretation
   gender_label            0.0           2                     0.0 profile-level (describes the ORIGINAL profile; constant across name swaps)
ethnicity_label            0.0           3                     0.0 profile-level (describes the ORIGINAL profile; constant across name swaps)
  -> gender_label: profile-level (describes the ORIGINAL profile; constant across name swaps)
  -> ethnicity_label: profile-level (describes the ORIGINAL profile; constant across name swaps)

=== fairness_comparison ===
      algorithm dataset                 target      group_attribute   metric_basis  accuracy  accuracy_ci_low  accuracy_ci_high  fairness_measure  fairness_ci_low  fairness_ci_high  n_rows  n_profiles
       baseline  kaggle         category_label intersectional_group       a

In [ ]:
!pytest /content/drive/MyDrive/bias_detection_project/tests/test_fairness_auditor.py -q

...............                                                          [100%]
15 passed in 6.30s


----------------------------------------------------------------------------------------------------------

In [ ]:
# === Colab cell: Report dashboard + static HTML report ===
# One-time, if ipywidgets is missing:  !pip -q install ipywidgets
import importlib
import os
import sys

import pandas as pd

PROJECT_ROOT = "/content/drive/MyDrive/bias_detection_project/"
sys.path.insert(0, os.path.join(PROJECT_ROOT, "code"))
import report_dashboard
importlib.reload(report_dashboard)
from report_dashboard import ReportDashboard

csv_path = os.path.join(PROJECT_ROOT, "reports", "fairness_comparison.csv")
html_path = os.path.join(PROJECT_ROOT, "reports", "fairness_report.html")

comparison = pd.read_csv(csv_path)
print(f"Loaded {len(comparison)} rows from {csv_path}")
print("Paired-difference columns present:",
      all(c in comparison.columns for c in ("fairness_diff", "fairness_diff_low", "fairness_diff_high")))

dash = ReportDashboard()

# 1) Interactive view (pick dataset / scores / comparison in the dropdowns)
dash.render_dashboard(comparison)

# 2) Static, shareable report (every combination, charts embedded)
saved = dash.export_html_report(comparison, html_path)
print("\n" + "=" * 70)
print("HTML report saved to:")
print(saved)
print("=" * 70)

Loaded 30 rows from /content/drive/MyDrive/bias_detection_project/reports/fairness_comparison.csv
Paired-difference columns present: False



HTML report saved to:
/content/drive/MyDrive/bias_detection_project/reports/fairness_report.html


--------------------------------------------------------------------------------------------------

In [ ]:
# ---- Colab cell: HRAgentInterceptor demo (fully independent) -----------------
# Loads the prepared data and thresholds itself (data_prep.load_prepared), uses
# the shared train/test split from data/processed/splits.csv, fits ONE
# BaselineScorer (TF-IDF + logistic regression, quick) on the TRAIN rows, and
# audits 200 held-out TEST profiles. No ImprovedScorer / BERTidRAAF training.
import sys, logging
import pandas as pd

PROJECT_ROOT = "/content/drive/MyDrive/bias_detection_project/"
DEMO_TARGET = None            # None = first FairCV target in the thresholds file
N_PROFILES = 200

sys.path.insert(0, PROJECT_ROOT + "code")
from data_prep import load_prepared
from baseline_scorer import BaselineScorer
from hr_agent_interceptor import HRAgentInterceptor, audited

logging.basicConfig(level=logging.WARNING)

# --- data, thresholds, split (same loader the pipeline uses) -----------------
all_df, THR = load_prepared(PROJECT_ROOT, apply_scrub=True)
is_fc = all_df["source_dataset"].astype(str).str.lower().str.contains("faircv")
train_df = all_df[all_df["split"] == "train"]
test_fc = all_df[(all_df["split"] == "test") & is_fc]
assert not set(train_df["original_id"]) & set(test_fc["original_id"]), "train/test overlap"

DEMO_TARGET = DEMO_TARGET or list(THR)[0]
print(f"FairCV targets in thresholds: {list(THR)}")
print(f"Baseline demo: BaselineScorer fitted on the train split for target = {DEMO_TARGET!r} "
      f"(faircv_threshold={THR[DEMO_TARGET]})")

# --- the baseline reads its label from `suitability_score`; load_prepared gives -
# one score column per target (the THR keys), so expose the chosen one under
# that name on BOTH frames. Kaggle rows have no FairCV score (NaN) and are ignored.
train_df = train_df.assign(suitability_score=train_df[DEMO_TARGET])
test_fc = test_fc.assign(suitability_score=test_fc[DEMO_TARGET])

# --- fit the baseline on TRAIN only ------------------------------------------
baseline = BaselineScorer(faircv_threshold=THR[DEMO_TARGET])
baseline.fit(train_df)

# --- 200 whole original_id values from TEST (all name variants kept together) -
ids = test_fc["original_id"].drop_duplicates()
sample_ids = ids.sample(n=min(N_PROFILES, len(ids)), random_state=42)
sample = test_fc[test_fc["original_id"].isin(sample_ids)].copy()
assert set(sample["split"]) == {"test"}
print(f"Sample: {sample['original_id'].nunique()} profiles, {len(sample)} rows (test split only)")

# (a) direct instantiation + .intercept()
direct = HRAgentInterceptor(baseline, fairness_threshold=0.8)
res_a = direct.intercept(sample, "faircv", target=DEMO_TARGET)
print(f"(a) direct     flagged={res_a['flagged']}  fairness_measure={res_a['fairness_measure']:.4f}"
      f"  CI(point, lo, hi)={tuple(round(x, 4) for x in res_a['fairness_ci'])}")

# (b) host app uses a plain function; the decorator audits it
@audited(baseline, threshold=0.8, dataset_type="faircv", target=DEMO_TARGET)
def mock_hr_agent(resumes):
    # the host's own logic; returns whatever it normally returns
    return {"n_resumes": len(resumes)}

agent_out = mock_hr_agent(sample)
res_b = mock_hr_agent.last_audit
print(f"(b) decorator  flagged={res_b['flagged']}  fairness_measure={res_b['fairness_measure']:.4f}"
      f"  (agent returned {agent_out})")

same = (res_a["flagged"] == res_b["flagged"]
        and abs(res_a["fairness_measure"] - res_b["fairness_measure"]) < 1e-12)
print("MATCH: direct and decorator results agree" if same
      else "MISMATCH: direct and decorator results differ")

# (c) OPTIONAL: only already-fitted Improved / BERTidRAAF-Lite scorers in memory
found = {n: globals()[n] for n in ("improved", "joint_scorer", "lite_scorer", "control")
         if n in globals() and globals()[n] is not None}
if not found:
    print("(c) skipped: no fitted ImprovedScorer/BERTidRAAFLiteScorer in memory.")
else:
    rows = []
    for var, scorer in found.items():
        res = HRAgentInterceptor(scorer, 0.8).intercept(sample, "faircv")  # all targets
        for t, m in res["fairness_measure"].items():
            rows.append({"variable": var, "scorer": res["scorer"], "target": t,
                         "fairness_measure": round(m, 4), "flagged": res["flagged"][t]})
    print(pd.DataFrame(rows).to_string(index=False))

FairCV targets in thresholds: ['score_blind', 'score_biased_gender', 'score_biased_ethnicity']
Baseline demo: BaselineScorer fitted on the train split for target = 'score_blind' (faircv_threshold=0.41373492308688775)
Sample: 200 profiles, 800 rows (test split only)
(a) direct     flagged=False  fairness_measure=0.9828  CI(point, lo, hi)=(0.9828, 0.9612, 1.0)
(b) decorator  flagged=False  fairness_measure=0.9828  (agent returned {'n_resumes': 800})
MATCH: direct and decorator results agree
(c) skipped: no fitted ImprovedScorer/BERTidRAAFLiteScorer in memory.


In [ ]:
!cd /content/drive/MyDrive/bias_detection_project && PYTHONPATH=code python -m pytest tests/test_hr_agent_interceptor.py

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0
rootdir: /content/drive/MyDrive/bias_detection_project
plugins: platformdirs-4.13.5, typeguard-4.6.0, langsmith-0.14.12, anyio-4.15.1
collected 15 items                                                             

tests/test_hr_agent_interceptor.py ...............                       [100%]

============================== 15 passed in 1.82s ==============================


-------------------------------------------------------------------------------------------------------

In [ ]:
import sys

PROJECT_ROOT = "/content/drive/MyDrive/bias_detection_project/"
sys.path.insert(0, PROJECT_ROOT + "code")

from run_full_pipeline import run_pipeline

result = run_pipeline(
    project_root=PROJECT_ROOT,
    force=False,
    joint_test=globals().get("joint_test")
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

... (2 outputs left out)

EmbeddingEngine: distilbert-base-uncased on cuda (hidden=768, max_length=512, fp16=True)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


EmbeddingEngine: distilbert-base-uncased on cuda (hidden=768, max_length=512, fp16=True)
       attribute                                                                                                                       improved                                                                                                                        control           status
  aux_reg_weight                                                                                                                            1.0                                                                                                                            1.0             same
     backbone_lr                                                                                                                          3e-05                                                                                                                          3e-05             same
      chunk_long                               

In [ ]:
import sys

PROJECT_ROOT = "/content/drive/MyDrive/bias_detection_project/"
sys.path.insert(0, PROJECT_ROOT + "code")

from validate_pipeline import validate_pipeline

validation = validate_pipeline(
    project_root=PROJECT_ROOT,
    pipeline_result=globals().get("result")
)

INGEST profiles: total=26484, FairCV=24000, Kaggle=2484; counterfactual rows=105936
PASS | 1 ingestion size | expected ~26481-26484; actual 26484
PASS | 1 expansion size | actual 105936, expected 105936
PASS | 1 native FairCV profile splits | {'test': 4800, 'train': 17280, 'val': 1920}
PASS | 1 identical prediction keys and target row counts | {'baseline': 58596, 'bertidraaf_lite': 58596, 'improved': 58596}
PASS | 2 baseline: no prediction/label NaNs
PASS | 3 baseline: all prediction files nonempty
PASS | 2 bertidraaf_lite: no prediction/label NaNs
PASS | 3 bertidraaf_lite: all prediction files nonempty
PASS | 2 improved: no prediction/label NaNs
PASS | 3 improved: all prediction files nonempty
PASS | 2 fairness_measure no NaNs
PASS | 3 fairness_measure in [0,1]
PASS | 4 point confidence intervals enclose estimates | [('accuracy', 0), ('fairness_measure', 0)]
PASS | 4 paired confidence intervals enclose differences | [('fairness_diff', 0)]
ACCURACY kaggle/category_label: baseline=0.638

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


EmbeddingEngine: distilbert-base-uncased on cuda (hidden=768, max_length=512, fp16=True)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


EmbeddingEngine: distilbert-base-uncased on cuda (hidden=768, max_length=512, fp16=True)
       attribute                                                                                                                       improved                                                                                                                        control           status
  aux_reg_weight                                                                                                                            1.0                                                                                                                            1.0             same
     backbone_lr                                                                                                                          3e-05                                                                                                                          3e-05             same
      chunk_long                               

----------------------------------------------------------------------------------------------------------------

In [ ]:

# ================================================================
# ARTIFACT RECOVERY + FULL PIPELINE
# Retrains only models missing deployable checkpoints.
# Uses existing datasets, native splits, thresholds and predictions.
# ================================================================

from google.colab import drive
drive.mount("/content/drive")

import sys
import json
import gc
import logging
import importlib
from pathlib import Path

import pandas as pd
import torch
import joblib

# ------------------------------------------------
# 1. PROJECT CONFIGURATION
# ------------------------------------------------

PROJECT_ROOT = "/content/drive/MyDrive/bias_detection_project/"

ROOT = Path(PROJECT_ROOT)
CODE = ROOT / "code"
PROC = ROOT / "data" / "processed"
ART = ROOT / "artifacts"

sys.path.insert(0, str(CODE))

ART.mkdir(parents=True, exist_ok=True)

MODELS = (
    "baseline",
    "improved",
    "bertidraaf_lite"
)

TARGETS = (
    "score_blind",
    "score_biased_gender",
    "score_biased_ethnicity"
)

# Set True only if you intentionally want to
# retrain all models, regardless of existing artifacts.
RETRAIN_ALL = False

# Checkpoints need all files, not merely a manifest.
ARTIFACT_FILES = {
    "baseline": [
        ART / "baseline" / task / "model.joblib"
        for task in ("kaggle", *TARGETS)
    ],
    "improved": [
        ART / "improved" / "weights.pt",
        ART / "improved" / "term_weights.joblib",
        ART / "improved" / "manifest.json"
    ],
    "bertidraaf_lite": [
        ART / "bertidraaf_lite" / "weights.pt",
        ART / "bertidraaf_lite" / "term_weights.joblib",
        ART / "bertidraaf_lite" / "manifest.json"
    ]
}

def exists_nonempty(path):
    return path.is_file() and path.stat().st_size > 0

def artifact_complete(model):
    return all(
        exists_nonempty(path)
        for path in ARTIFACT_FILES[model]
    )

# ------------------------------------------------
# 2. IDENTIFY REQUIRED TRAINING
# ------------------------------------------------

print("=" * 65)
print("BIAS DETECTION - ARTIFACT RECOVERY")
print("=" * 65)

force_stages = []

for model in MODELS:
    complete = artifact_complete(model)

    if RETRAIN_ALL or not complete:
        force_stages.append(model)
        print(f"RETRAIN REQUIRED | {model}")
    else:
        print(f"REUSE ARTIFACT   | {model}")

print("\nStages selected:", force_stages or "none")

# ------------------------------------------------
# 3. PROTECT EXISTING LOCKED KAGGLE PREDICTIONS
# ------------------------------------------------

locked_path = (
    PROC / "improved_predictions_kaggle_locked.csv"
)

if not exists_nonempty(locked_path):
    raise FileNotFoundError(
        "The locked improved Kaggle prediction file "
        "is missing. Restore it before training."
    )

# Take a SHA256 fingerprint to verify that
# the locked prediction CSV is never modified.
import hashlib

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()

locked_before = file_sha256(locked_path)

print("\nLocked Kaggle CSV protected.")
print("SHA256:", locked_before)

# ------------------------------------------------
# 4. CONFIGURE MATCHED PAIR MEMORY HANDLING
# ------------------------------------------------

# build_matched_pair() performs the same configuration
# checks when engine_factory=None. This allows the two
# engines to load lazily rather than allocating both
# DistilBERT backbones during a configuration-only check.

import bertidraaf_lite_scorer as lite_module

original_builder = lite_module.build_matched_pair

def memory_safe_matched_pair(
    common_kwargs,
    ibs_weight_mode,
    engine_factory=None,
    **kwargs
):
    return original_builder(
        common_kwargs,
        ibs_weight_mode,
        engine_factory=None,
        **kwargs
    )

lite_module.build_matched_pair = memory_safe_matched_pair

# ------------------------------------------------
# 5. FIX PREDICTION SCHEMA ASSEMBLY
# ------------------------------------------------

import run_full_pipeline as pipeline_module
importlib.reload(pipeline_module)

def assemble_predictions(
    scorer,
    test,
    reference_columns=None
):
    src = test["source_dataset"].astype(str).str.lower()

    kaggle = test[
        src.str.contains("kaggle", regex=False)
    ]

    faircv = test[
        src.str.contains("faircv", regex=False)
    ]

    # Kaggle predictions from the freshly trained scorer
    k = scorer.predict_kaggle(kaggle)

    k.insert(0, "dataset", "kaggle")
    k.insert(1, "target", "category_label")

    # All three FairCV targets
    f = scorer.predict_faircv(faircv)

    if "dataset" in f.columns:
        f = f.drop(columns=["dataset"])

    f.insert(0, "dataset", "faircv")

    combined = pd.concat(
        [k, f],
        ignore_index=True
    )

    required = {
        "dataset",
        "target",
        "resume_id",
        "original_id",
        "true_label",
        "predicted_label",
        "confidence"
    }

    missing = required - set(combined.columns)

    if missing:
        raise ValueError(
            f"Missing prediction columns: {missing}"
        )

    if reference_columns is not None:
        columns = list(reference_columns)

        # Optional model-specific columns are allowed.
        # Required evaluation columns must match.
        missing_reference = set(columns) - set(combined.columns)

        if missing_reference:
            raise ValueError(
                "Control prediction schema does not match "
                f"the improved scorer: {missing_reference}"
            )

        combined = combined[columns]

    return combined

pipeline_module._assemble = assemble_predictions

# ------------------------------------------------
# 6. GPU CLEANUP BEFORE TRAINING
# ------------------------------------------------

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

    print("\nGPU:", torch.cuda.get_device_name(0))
    print(
        "Allocated memory:",
        round(
            torch.cuda.memory_allocated() / 1024**3,
            2
        ),
        "GB"
    )
else:
    print(
        "\nWARNING: No CUDA GPU found. "
        "Neural training may be much slower."
    )

# ------------------------------------------------
# 7. RUN THE PIPELINE
# ------------------------------------------------

# Important:
# joint_test=None is deliberate. We must FIT and
# SAVE the neural model when its artifact is absent.
# Reusing an in-memory predictions DataFrame would
# skip the model training required for a checkpoint.

print("\n" + "=" * 65)
print("RUNNING REQUIRED TRAINING STAGES")
print("=" * 65)

result = pipeline_module.run_pipeline(
    project_root=PROJECT_ROOT,
    force=False,
    force_stages=force_stages,
    joint_test=None,
    n_iterations=1000
)

# ------------------------------------------------
# 8. VERIFY LOCKED KAGGLE CSV
# ------------------------------------------------

locked_after = file_sha256(locked_path)

assert locked_before == locked_after, (
    "ERROR: Locked Kaggle CSV was modified!"
)

print("\nPASS | Locked Kaggle predictions unchanged")

# ------------------------------------------------
# 9. VERIFY BASELINE ARTIFACTS
# ------------------------------------------------

print("\n" + "=" * 65)
print("CHECKING BASELINE ARTIFACTS")
print("=" * 65)

verification = {}

for task in ("kaggle", *TARGETS):
    path = ART / "baseline" / task / "model.joblib"

    if not exists_nonempty(path):
        verification[f"baseline/{task}"] = False
        print(f"FAIL | {task}: checkpoint missing")
        continue

    try:
        payload = joblib.load(path)

        assert "vectorizer" in payload
        assert "classifier" in payload
        assert hasattr(payload["vectorizer"], "transform")
        assert hasattr(payload["classifier"], "predict")

        verification[f"baseline/{task}"] = True

        print(
            f"PASS | {task}: "
            f"{path.stat().st_size / 1024**2:.2f} MB"
        )

        del payload

    except Exception as e:
        verification[f"baseline/{task}"] = False
        print(f"FAIL | {task}: {e}")

# ------------------------------------------------
# 10. VERIFY NEURAL CHECKPOINTS
# ------------------------------------------------

print("\n" + "=" * 65)
print("CHECKING BERT MODEL ARTIFACTS")
print("=" * 65)

for model in ("improved", "bertidraaf_lite"):

    folder = ART / model
    status = True

    try:
        for path in ARTIFACT_FILES[model]:
            if not exists_nonempty(path):
                raise FileNotFoundError(path)

        manifest = json.loads(
            (folder / "manifest.json").read_text()
        )

        checkpoint = torch.load(
            folder / "weights.pt",
            map_location="cpu",
            weights_only=True
        )

        lookup = joblib.load(
            folder / "term_weights.joblib"
        )

        assert manifest["model_kind"] == model

        assert checkpoint["head"]
        assert checkpoint["backbone"]
        assert len(lookup) > 0

        assert set(manifest["faircv_targets"]) == set(TARGETS)

        assert manifest["classes"]

        print(f"\nPASS | {model}")
        print(
            "  Head parameters:",
            len(checkpoint["head"])
        )
        print(
            "  Backbone parameters:",
            len(checkpoint["backbone"])
        )
        print(
            "  Term weights:",
            len(lookup)
        )
        print(
            "  Kaggle classes:",
            len(manifest["classes"])
        )
        print(
            "  Model weight mode:",
            manifest["weight_mode"]
        )

        del checkpoint, lookup

    except Exception as e:
        status = False
        print(f"\nFAIL | {model}: {repr(e)}")

    verification[model] = status

    gc.collect()

# ------------------------------------------------
# 11. CREATE APPLICATION ARTIFACT MANIFEST
# ------------------------------------------------

inventory = {
    "project": "bias_detection_project",
    "artifact_version": 1,
    "models": {},
    "locked_kaggle_sha256": locked_after,
    "application_inference": True
}

for model in MODELS:

    inventory["models"][model] = {
        "available": all(
            verification.get(
                f"baseline/{task}",
                False
            )
            for task in ("kaggle", *TARGETS)
        ) if model == "baseline"
        else verification.get(model, False),
        "artifact_path": str(ART / model),
        "prediction_csv": str(
            PROC / f"{model}_predictions.csv"
        )
    }

inventory_path = ART / "artifact_inventory.json"

inventory_path.write_text(
    json.dumps(inventory, indent=2)
)

# ------------------------------------------------
# 12. FINAL ARTIFACT VALIDATION
# ------------------------------------------------

print("\n" + "=" * 65)
print("FINAL ARTIFACT SUMMARY")
print("=" * 65)

for name, passed in verification.items():
    print(
        f"{'PASS' if passed else 'FAIL'} | {name}"
    )

failed = [
    name
    for name, passed in verification.items()
    if not passed
]

if failed:
    print(
        "\nOVERALL FAIL | Missing or invalid artifacts:",
        failed
    )
    raise RuntimeError(
        "Some artifacts failed verification. "
        "Review the failures above."
    )

print("\nOVERALL PASS | All model artifacts saved")
print("Artifact root:", ART)
print("Inventory:", inventory_path)

print("\nSaved files:")
for file in sorted(ART.rglob("*")):
    if file.is_file():
        print(
            file.relative_to(ART),
            f"({file.stat().st_size / 1024**2:.2f} MB)"
        )


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BIAS DETECTION - ARTIFACT RECOVERY
REUSE ARTIFACT   | baseline
REUSE ARTIFACT   | improved
RETRAIN REQUIRED | bertidraaf_lite

Stages selected: ['bertidraaf_lite']

Locked Kaggle CSV protected.
SHA256: 5bb028c95b11fec6b1c6e04d0d4051f2dd258b22e2ac5a88e6c42a7d573f5da5

GPU: Tesla T4
Allocated memory: 0.0 GB

RUNNING REQUIRED TRAINING STAGES
       attribute                                                                                                                       improved                                                                                                                        control    status
  aux_reg_weight                                                                                                                            1.0                                                                                                          

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


EmbeddingEngine: distilbert-base-uncased on cuda (hidden=768, max_length=512, fp16=True)
IDF: N=77068 docs, 36032 terms | oov_weight = 4.406 [token-median IDF] (token-median 4.406, type-median 10.055, Eq.1 df=0 value 12.252)
Fine-tuning backbone (lr 3e-05) + gate/heads (lr 0.001): 665 steps/epoch x 6 epochs
Epoch 1/6 | loss 0.7735 | kaggle CE 1.8708 | faircv BCE 0.5195 | mean gate 0.402 | val kaggle acc 0.8125 | val faircv acc 0.7670 | 1.4 min
Epoch 2/6 | loss 0.5178 | kaggle CE 0.6948 | faircv BCE 0.4768 | mean gate 0.538 | val kaggle acc 0.8196 | val faircv acc 0.7828 | 1.3 min
Epoch 3/6 | loss 0.4647 | kaggle CE 0.4405 | faircv BCE 0.4703 | mean gate 0.556 | val kaggle acc 0.8498 | val faircv acc 0.7811 | 1.3 min
Epoch 4/6 | loss 0.4305 | kaggle CE 0.2798 | faircv BCE 0.4654 | mean gate 0.574 | val kaggle acc 0.8397 | val faircv acc 0.7760 | 1.3 min
Epoch 5/6 | loss 0.4091 | kaggle CE 0.1790 | faircv BCE 0.4624 | mean gate 0.577 | val kaggle acc 0.8498 | val faircv acc 0.7840 | 1.3 